# FAITH-Detect — GPU-only reviewer revisions (Colab T4/A100)

Runs the two experiments that don't fit the 8 GB local box and prints every result **inline**, so
the executed notebook itself carries the numbers — just **Run all**, then send the notebook back.

**A. SoftReg $\lambda$ sweep** — reliance vs. accuracy frontier (full scale, roberta-base).  
**B. Full-scale same-domain multi-generator** — isolate generator shift from domain shift.

### Before you start — build the code zip from your *local* repo (it has the updated scripts):
```bash
cd /Users/shiva
zip -r FAITH-Detect.zip FAITH-Detect \
  -x '*/.git/*' '*/results/cells/*' '*.pt' '*/data/*' '*/results_colab/*' '*/paper/*.pdf'
```
Then run the cells below; you'll be asked to upload `FAITH-Detect.zip` and `data/all_data.csv`.


In [ ]:
!nvidia-smi -L

In [ ]:
# 1) Dependencies
!pip -q install 'transformers>=4.40' 'datasets>=2.19' 'huggingface_hub>=0.23' \
    scikit-learn pandas pyarrow nltk spacy matplotlib
import nltk
for r in ['stopwords','punkt','punkt_tab','wordnet','omw-1.4']:
    try: nltk.download(r, quiet=True)
    except Exception as e: print('nltk', r, e)
import os; os.system('python -m spacy download en_core_web_sm -q')

In [ ]:
# 2) Upload the code zip (FAITH-Detect.zip built above). Re-run after edits: %cd /content; !rm -rf FAITH-Detect
import os, zipfile
from google.colab import files
if not os.path.exists('FAITH-Detect'):
    print('Upload FAITH-Detect.zip:')
    up = files.upload(); name = next(iter(up))
    with zipfile.ZipFile(name) as z: z.extractall('.')
%cd FAITH-Detect
import sys; sys.path.insert(0, 'src')

In [ ]:
# Sync the revised modules from embedded copies (immune to a stale uploaded zip).
import base64, pathlib, os
_SYNC = {
  'src/faithdetect/data/collate.py': "IiIiQmF0Y2ggY29sbGF0aW9uIHdpdGggZnVuY3Rpb24td29yZCBtYXNraW5nLgoKQSBzaW5nbGUgQ29sbGF0b3Igc2VydmVzIGV2ZXJ5IG1vZGVsIHZhcmlhbnQgdmlhIGBgbWFza19tb2RlYGA6CiAgKiBiYXNlbGluZSAvIHNvZnRyZWcgIC0+IG1hc2tfbW9kZT0nbm9uZScgIChmd19tYXNrIHN0aWxsIHJldHVybmVkOyBzb2Z0cmVnIGNvbnN1bWVzIGl0KS4KICAqIGhhcmRtYXNrICAgICAgICAgICAgLT4gbWFza19tb2RlPSdoYXJkbWFzayc6IEZXIHRva2VuIGlkcyByZXBsYWNlZCBieSBhIHNpbmdsZSBzaGFyZWQKICAgICAgICAgICAgICAgICAgICAgICAgICAgcGxhY2Vob2xkZXIgKFtGVU5DXSkgLT4gaWRlbnRpdHkgZXJhc2VkLCBzbG90IHBvc2l0aW9uK2NvdW50IGtlcHQuCiAgKiBkZWxldGlvbiAgICAgICAgICAgIC0+IG1hc2tfbW9kZT0nZGVsZXRpb24nOiBGVyB0b2tlbnMgcmVtb3ZlZCBlbnRpcmVseSAtPiBpZGVudGl0eSBBTkQKICAgICAgICAgICAgICAgICAgICAgICAgICAgc2xvdCByZW1vdmVkIChzZXF1ZW5jZSBzaG9ydGVucykuICJUcnVseSByZW1vdmVzIGZ1bmN0aW9uIHdvcmRzLiIKICAqIHJhbmRvbSAgICAgICAgICAgICAgLT4gbWFza19tb2RlPSdyYW5kb20nOiBlYWNoIEZXIHRva2VuIHJlcGxhY2VkIGJ5IGEgY29udGVudC1oYXNoLXNlZWRlZAogICAgICAgICAgICAgICAgICAgICAgICAgICByYW5kb20gdm9jYWJ1bGFyeSBpZCAtPiBpZGVudGl0eSBlcmFzZWQgYW5kIHRoZSBjb25zaXN0ZW50IFtGVU5DXQogICAgICAgICAgICAgICAgICAgICAgICAgICBzbG90LW1hcmtlciByZW1vdmVkLCB3aGlsZSB0aGUgcGVydHVyYmVkIHBvc2l0aW9ucy9jb3VudCBhcmUga2VwdC4KVGhlIGxhc3QgdHdvIGFyZSBjb250cm9scyB0aGF0IHNlcGFyYXRlIGlkZW50aXR5LCBzbG90LW1hcmtlciBhbmQgcG9zaXRpb24vc3ludGF4IGVmZmVjdHMuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgpmcm9tIC4uZnVuY3Rpb25fd29yZHMgaW1wb3J0IEZ1bmN0aW9uV29yZFNldCwgZnVuY3Rpb25fd29yZF90b2tlbl9tYXNrLCBhcHBseV9oYXJkX21hc2sKCgpjbGFzcyBDb2xsYXRvcjoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIHRva2VuaXplciwKICAgICAgICBmd19zZXQ6IEZ1bmN0aW9uV29yZFNldCwKICAgICAgICBtYXhfbGVuZ3RoOiBpbnQgPSAyNTYsCiAgICAgICAgaGFyZF9tYXNrOiBib29sID0gRmFsc2UsCiAgICAgICAgcGxhY2Vob2xkZXJfaWQ6IGludCB8IE5vbmUgPSBOb25lLAogICAgICAgIHBhZGRpbmc6IHN0ciB8IGJvb2wgPSBUcnVlLAogICAgICAgIG1hc2tfbW9kZTogc3RyID0gImF1dG8iLAogICAgKToKICAgICAgICBzZWxmLnRva2VuaXplciA9IHRva2VuaXplcgogICAgICAgIHNlbGYuZndfc2V0ID0gZndfc2V0CiAgICAgICAgc2VsZi5tYXhfbGVuZ3RoID0gbWF4X2xlbmd0aAogICAgICAgIHNlbGYuaGFyZF9tYXNrID0gaGFyZF9tYXNrCiAgICAgICAgc2VsZi5wbGFjZWhvbGRlcl9pZCA9IHBsYWNlaG9sZGVyX2lkCiAgICAgICAgc2VsZi5wYWRkaW5nID0gcGFkZGluZwogICAgICAgICMgYmFjay1jb21wYXQ6ICdhdXRvJyBkZXJpdmVzIHRoZSBtb2RlIGZyb20gdGhlIGhhcmRfbWFzayBmbGFnLgogICAgICAgIHNlbGYubWFza19tb2RlID0gKCJoYXJkbWFzayIgaWYgaGFyZF9tYXNrIGVsc2UgIm5vbmUiKSBpZiBtYXNrX21vZGUgPT0gImF1dG8iIGVsc2UgbWFza19tb2RlCiAgICAgICAgaWYgc2VsZi5tYXNrX21vZGUgaW4gKCJoYXJkbWFzayIsKSBhbmQgcGxhY2Vob2xkZXJfaWQgaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaGFyZG1hc2sgcmVxdWlyZXMgYSBwbGFjZWhvbGRlcl9pZCIpCiAgICAgICAgc2VsZi5fcGFkX2lkID0gdG9rZW5pemVyLnBhZF90b2tlbl9pZAogICAgICAgIHNlbGYuX3NwZWNpYWwgPSBzZXQodG9rZW5pemVyLmFsbF9zcGVjaWFsX2lkcykKICAgICAgICBzZWxmLl92b2NhYiA9IGxlbih0b2tlbml6ZXIpCgogICAgZGVmIF9yYW5kb21fcmVwbGFjZShzZWxmLCBpbnB1dF9pZHM6IG5wLm5kYXJyYXksIGZ3X21hc2s6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgb3V0ID0gaW5wdXRfaWRzLmNvcHkoKQogICAgICAgIGZvciByIGluIHJhbmdlKG91dC5zaGFwZVswXSk6CiAgICAgICAgICAgIGNvbHMgPSBucC53aGVyZShmd19tYXNrW3JdKVswXQogICAgICAgICAgICBpZiBjb2xzLnNpemUgPT0gMDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHJuZyA9IG5wLnJhbmRvbS5SYW5kb21TdGF0ZShhYnMoaGFzaCh0dXBsZShpbnQoeCkgZm9yIHggaW4gaW5wdXRfaWRzW3JdKSkpICUgKDIqKjMyKSkKICAgICAgICAgICAgZm9yIGMgaW4gY29sczoKICAgICAgICAgICAgICAgIHRpZCA9IGludChybmcucmFuZGludCgwLCBzZWxmLl92b2NhYikpCiAgICAgICAgICAgICAgICB3aGlsZSB0aWQgaW4gc2VsZi5fc3BlY2lhbCBvciB0aWQgPT0gc2VsZi5wbGFjZWhvbGRlcl9pZDoKICAgICAgICAgICAgICAgICAgICB0aWQgPSBpbnQocm5nLnJhbmRpbnQoMCwgc2VsZi5fdm9jYWIpKQogICAgICAgICAgICAgICAgb3V0W3IsIGNdID0gdGlkCiAgICAgICAgcmV0dXJuIG91dAoKICAgIGRlZiBfZGVsZXRlKHNlbGYsIGlucHV0X2lkczogbnAubmRhcnJheSwgYXR0bjogbnAubmRhcnJheSwgZndfbWFzazogbnAubmRhcnJheSk6CiAgICAgICAgcm93c19pZHMsIHJvd3NfbGVuID0gW10sIFtdCiAgICAgICAgZm9yIHIgaW4gcmFuZ2UoaW5wdXRfaWRzLnNoYXBlWzBdKToKICAgICAgICAgICAga2VlcCA9IChhdHRuW3JdID09IDEpICYgKH5md19tYXNrW3JdKQogICAgICAgICAgICBrZXB0ID0gaW5wdXRfaWRzW3JdW2tlZXBdCiAgICAgICAgICAgIHJvd3NfaWRzLmFwcGVuZChrZXB0KQogICAgICAgICAgICByb3dzX2xlbi5hcHBlbmQobGVuKGtlcHQpKQogICAgICAgIHdpZHRoID0gbWF4KHJvd3NfbGVuKSBpZiByb3dzX2xlbiBlbHNlIDEKICAgICAgICBuZXdfaWRzID0gbnAuZnVsbCgoaW5wdXRfaWRzLnNoYXBlWzBdLCB3aWR0aCksIHNlbGYuX3BhZF9pZCwgZHR5cGU9aW5wdXRfaWRzLmR0eXBlKQogICAgICAgIG5ld19hdHRuID0gbnAuemVyb3MoKGlucHV0X2lkcy5zaGFwZVswXSwgd2lkdGgpLCBkdHlwZT1hdHRuLmR0eXBlKQogICAgICAgIGZvciByLCBrZXB0IGluIGVudW1lcmF0ZShyb3dzX2lkcyk6CiAgICAgICAgICAgIG5ld19pZHNbciwgOiBsZW4oa2VwdCldID0ga2VwdAogICAgICAgICAgICBuZXdfYXR0bltyLCA6IGxlbihrZXB0KV0gPSAxCiAgICAgICAgcmV0dXJuIG5ld19pZHMsIG5ld19hdHRuCgogICAgZGVmIF9fY2FsbF9fKHNlbGYsIGJhdGNoOiBsaXN0W2RpY3RdKSAtPiBkaWN0OgogICAgICAgIHRleHRzID0gW2JbInRleHQiXSBmb3IgYiBpbiBiYXRjaF0KICAgICAgICBsYWJlbHMgPSBbYlsibGFiZWwiXSBmb3IgYiBpbiBiYXRjaF0KICAgICAgICBpbmRpY2VzID0gW2IuZ2V0KCJpbmRleCIsIC0xKSBmb3IgYiBpbiBiYXRjaF0KICAgICAgICBlbmMsIGZ3X21hc2sgPSBmdW5jdGlvbl93b3JkX3Rva2VuX21hc2soCiAgICAgICAgICAgIHRleHRzLCBzZWxmLnRva2VuaXplciwgc2VsZi5md19zZXQsIHNlbGYubWF4X2xlbmd0aCwgcGFkZGluZz1zZWxmLnBhZGRpbmcKICAgICAgICApCiAgICAgICAgaW5wdXRfaWRzID0gbnAuYXJyYXkoZW5jWyJpbnB1dF9pZHMiXSkKICAgICAgICBhdHRuID0gbnAuYXJyYXkoZW5jWyJhdHRlbnRpb25fbWFzayJdKQogICAgICAgIGZ3X21hc2sgPSBucC5hcnJheShmd19tYXNrKQoKICAgICAgICBpZiBzZWxmLm1hc2tfbW9kZSA9PSAiaGFyZG1hc2siOgogICAgICAgICAgICBpbnB1dF9pZHMgPSBhcHBseV9oYXJkX21hc2soaW5wdXRfaWRzLCBmd19tYXNrLCBzZWxmLnBsYWNlaG9sZGVyX2lkKQogICAgICAgIGVsaWYgc2VsZi5tYXNrX21vZGUgPT0gInJhbmRvbSI6CiAgICAgICAgICAgIGlucHV0X2lkcyA9IHNlbGYuX3JhbmRvbV9yZXBsYWNlKGlucHV0X2lkcywgZndfbWFzaykKICAgICAgICBlbGlmIHNlbGYubWFza19tb2RlID09ICJkZWxldGlvbiI6CiAgICAgICAgICAgIGlucHV0X2lkcywgYXR0biA9IHNlbGYuX2RlbGV0ZShpbnB1dF9pZHMsIGF0dG4sIGZ3X21hc2spCiAgICAgICAgICAgIGZ3X21hc2sgPSBucC56ZXJvc19saWtlKGlucHV0X2lkcywgZHR5cGU9Ym9vbCkKCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgImlucHV0X2lkcyI6IHRvcmNoLmFzX3RlbnNvcihpbnB1dF9pZHMsIGR0eXBlPXRvcmNoLmxvbmcpLAogICAgICAgICAgICAiYXR0ZW50aW9uX21hc2siOiB0b3JjaC5hc190ZW5zb3IoYXR0biwgZHR5cGU9dG9yY2gubG9uZyksCiAgICAgICAgICAgICJmd19tYXNrIjogdG9yY2guYXNfdGVuc29yKGZ3X21hc2ssIGR0eXBlPXRvcmNoLmJvb2wpLAogICAgICAgICAgICAibGFiZWxzIjogdG9yY2guYXNfdGVuc29yKGxhYmVscywgZHR5cGU9dG9yY2gubG9uZyksCiAgICAgICAgICAgICJpbmRleCI6IHRvcmNoLmFzX3RlbnNvcihpbmRpY2VzLCBkdHlwZT10b3JjaC5sb25nKSwKICAgICAgICB9Cg==",
  'src/faithdetect/train.py': "IiIiU2VlZGVkIHRyYWluaW5nIGxvb3AgKEFkYW1XICsgbGluZWFyIHdhcm11cCwgZWFybHkgc3RvcHBpbmcpIGZvciBhbGwgdGhyZWUgdmFyaWFudHMuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBjb3B5CmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXIKZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IGdldF9saW5lYXJfc2NoZWR1bGVfd2l0aF93YXJtdXAKCmZyb20gLmRhdGEuY29sbGF0ZSBpbXBvcnQgQ29sbGF0b3IKZnJvbSAuZGF0YS5tYWlkZV91cCBpbXBvcnQgVGV4dExhYmVsRGF0YXNldApmcm9tIC5mdW5jdGlvbl93b3JkcyBpbXBvcnQgRnVuY3Rpb25Xb3JkU2V0CmZyb20gLm1vZGVscyBpbXBvcnQgTW9kZWxDb25maWcsIFJldmlld0RldGVjdG9yLCBjb21wdXRlX2xvc3MKZnJvbSAudXRpbHMuc2VlZGluZyBpbXBvcnQgc2V0X3NlZWQsIHNlZWRfd29ya2VyCgoKQGRhdGFjbGFzcwpjbGFzcyBUcmFpbkNvbmZpZzoKICAgIGVwb2NoczogaW50ID0gNAogICAgYmF0Y2hfc2l6ZTogaW50ID0gMTYKICAgIGxyOiBmbG9hdCA9IDJlLTUKICAgIHdlaWdodF9kZWNheTogZmxvYXQgPSAwLjAxCiAgICB3YXJtdXBfcmF0aW86IGZsb2F0ID0gMC4xCiAgICBtYXhfZ3JhZF9ub3JtOiBmbG9hdCA9IDEuMAogICAgcGF0aWVuY2U6IGludCA9IDIgICAgICAgICAgIyBlYXJseS1zdG9wIHBhdGllbmNlIG9uIHZhbCBtYWNyby1GMQogICAgbnVtX3dvcmtlcnM6IGludCA9IDAKICAgIGxvZ19ldmVyeTogaW50ID0gMCAgICAgICAgICMgMCA9IHNpbGVudCBiYXRjaCBsb2dnaW5nCgoKZGVmIG1ha2VfY29sbGF0b3IoCiAgICBtb2RlbF9jZmc6IE1vZGVsQ29uZmlnLAogICAgdG9rZW5pemVyLAogICAgZnVuY19pZDogaW50LAogICAgZndfc2V0OiBGdW5jdGlvbldvcmRTZXQsCikgLT4gQ29sbGF0b3I6CiAgICAiIiJDb2xsYXRvciBjb25zaXN0ZW50IHdpdGggdGhlIHZhcmlhbnQuIGJhc2VsaW5lL3NvZnRyZWcgLT4gbm8gbWFza2luZzsgaGFyZG1hc2sgLT4KICAgIFtGVU5DXSBwbGFjZWhvbGRlcjsgZGVsZXRpb24gLT4gZHJvcCBGVyB0b2tlbnM7IHJhbmRvbSAtPiByYW5kb20tdG9rZW4gcGxhY2Vob2xkZXIuIiIiCiAgICBtb2RlID0geyJoYXJkbWFzayI6ICJoYXJkbWFzayIsICJkZWxldGlvbiI6ICJkZWxldGlvbiIsICJyYW5kb20iOiAicmFuZG9tIn0uZ2V0KAogICAgICAgIG1vZGVsX2NmZy52YXJpYW50LCAibm9uZSIKICAgICkKICAgIHJldHVybiBDb2xsYXRvcigKICAgICAgICB0b2tlbml6ZXI9dG9rZW5pemVyLAogICAgICAgIGZ3X3NldD1md19zZXQsCiAgICAgICAgbWF4X2xlbmd0aD1tb2RlbF9jZmcubWF4X2xlbmd0aCwKICAgICAgICBoYXJkX21hc2s9KG1vZGVsX2NmZy52YXJpYW50ID09ICJoYXJkbWFzayIpLAogICAgICAgIHBsYWNlaG9sZGVyX2lkPWZ1bmNfaWQsCiAgICAgICAgcGFkZGluZz1UcnVlLAogICAgICAgIG1hc2tfbW9kZT1tb2RlLAogICAgKQoKCmRlZiBfbW92ZShiYXRjaDogZGljdCwgZGV2aWNlKSAtPiBkaWN0OgogICAgcmV0dXJuIHtrOiAodi50byhkZXZpY2UpIGlmIHRvcmNoLmlzX3RlbnNvcih2KSBlbHNlIHYpIGZvciBrLCB2IGluIGJhdGNoLml0ZW1zKCl9CgoKQHRvcmNoLm5vX2dyYWQoKQpkZWYgX3ZhbF9tYWNyb19mMShtb2RlbCwgbG9hZGVyLCBkZXZpY2UpIC0+IGZsb2F0OgogICAgZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IGYxX3Njb3JlCgogICAgbW9kZWwuZXZhbCgpCiAgICBwcmVkcywgdHJ1ZXMgPSBbXSwgW10KICAgIGZvciBiYXRjaCBpbiBsb2FkZXI6CiAgICAgICAgYmF0Y2ggPSBfbW92ZShiYXRjaCwgZGV2aWNlKQogICAgICAgIGxvZ2l0cyA9IG1vZGVsKGlucHV0X2lkcz1iYXRjaFsiaW5wdXRfaWRzIl0sIGF0dGVudGlvbl9tYXNrPWJhdGNoWyJhdHRlbnRpb25fbWFzayJdKQogICAgICAgIHByZWRzLmV4dGVuZChsb2dpdHMuYXJnbWF4KDEpLmNwdSgpLm51bXB5KCkpCiAgICAgICAgdHJ1ZXMuZXh0ZW5kKGJhdGNoWyJsYWJlbHMiXS5jcHUoKS5udW1weSgpKQogICAgcmV0dXJuIGZsb2F0KGYxX3Njb3JlKHRydWVzLCBwcmVkcywgYXZlcmFnZT0ibWFjcm8iKSkKCgpkZWYgdHJhaW5fbW9kZWwoCiAgICBtb2RlbF9jZmc6IE1vZGVsQ29uZmlnLAogICAgdHJhaW5fY2ZnOiBUcmFpbkNvbmZpZywKICAgIHNwbGl0cywKICAgIHRva2VuaXplciwKICAgIGZ1bmNfaWQ6IGludCwKICAgIGZ3X3NldDogRnVuY3Rpb25Xb3JkU2V0LAogICAgZGV2aWNlLAogICAgc2VlZDogaW50LAopIC0+IHR1cGxlW1Jldmlld0RldGVjdG9yLCBkaWN0XToKICAgICIiIlRyYWluIG9uZSBtb2RlbCBhbmQgcmV0dXJuIChiZXN0X21vZGVsLCBoaXN0b3J5KS4iIiIKICAgIHNldF9zZWVkKHNlZWQpCiAgICBtb2RlbCA9IFJldmlld0RldGVjdG9yKG1vZGVsX2NmZywgdm9jYWJfc2l6ZT1sZW4odG9rZW5pemVyKSkudG8oZGV2aWNlKQogICAgY29sbGF0b3IgPSBtYWtlX2NvbGxhdG9yKG1vZGVsX2NmZywgdG9rZW5pemVyLCBmdW5jX2lkLCBmd19zZXQpCgogICAgZyA9IHRvcmNoLkdlbmVyYXRvcigpCiAgICBnLm1hbnVhbF9zZWVkKHNlZWQpCiAgICB0cmFpbl9sb2FkZXIgPSBEYXRhTG9hZGVyKAogICAgICAgIFRleHRMYWJlbERhdGFzZXQuZnJvbV9mcmFtZShzcGxpdHMudHJhaW4pLAogICAgICAgIGJhdGNoX3NpemU9dHJhaW5fY2ZnLmJhdGNoX3NpemUsIHNodWZmbGU9VHJ1ZSwgY29sbGF0ZV9mbj1jb2xsYXRvciwKICAgICAgICBudW1fd29ya2Vycz10cmFpbl9jZmcubnVtX3dvcmtlcnMsIHdvcmtlcl9pbml0X2ZuPXNlZWRfd29ya2VyLCBnZW5lcmF0b3I9ZywKICAgICkKICAgIHZhbF9sb2FkZXIgPSBEYXRhTG9hZGVyKAogICAgICAgIFRleHRMYWJlbERhdGFzZXQuZnJvbV9mcmFtZShzcGxpdHMudmFsKSwKICAgICAgICBiYXRjaF9zaXplPXRyYWluX2NmZy5iYXRjaF9zaXplLCBzaHVmZmxlPUZhbHNlLCBjb2xsYXRlX2ZuPWNvbGxhdG9yLAogICAgICAgIG51bV93b3JrZXJzPXRyYWluX2NmZy5udW1fd29ya2VycywKICAgICkKCiAgICBub19kZWNheSA9IFsiYmlhcyIsICJMYXllck5vcm0ud2VpZ2h0Il0KICAgIGdyb3VwZWQgPSBbCiAgICAgICAgeyJwYXJhbXMiOiBbcCBmb3IgbiwgcCBpbiBtb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCkgaWYgbm90IGFueShuZCBpbiBuIGZvciBuZCBpbiBub19kZWNheSldLAogICAgICAgICAid2VpZ2h0X2RlY2F5IjogdHJhaW5fY2ZnLndlaWdodF9kZWNheX0sCiAgICAgICAgeyJwYXJhbXMiOiBbcCBmb3IgbiwgcCBpbiBtb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCkgaWYgYW55KG5kIGluIG4gZm9yIG5kIGluIG5vX2RlY2F5KV0sCiAgICAgICAgICJ3ZWlnaHRfZGVjYXkiOiAwLjB9LAogICAgXQogICAgb3B0aW1pemVyID0gdG9yY2gub3B0aW0uQWRhbVcoZ3JvdXBlZCwgbHI9dHJhaW5fY2ZnLmxyKQogICAgdG90YWxfc3RlcHMgPSBtYXgoMSwgbGVuKHRyYWluX2xvYWRlcikgKiB0cmFpbl9jZmcuZXBvY2hzKQogICAgc2NoZWR1bGVyID0gZ2V0X2xpbmVhcl9zY2hlZHVsZV93aXRoX3dhcm11cCgKICAgICAgICBvcHRpbWl6ZXIsIGludCh0cmFpbl9jZmcud2FybXVwX3JhdGlvICogdG90YWxfc3RlcHMpLCB0b3RhbF9zdGVwcwogICAgKQoKICAgIGhpc3RvcnkgPSB7InRyYWluX2xvc3MiOiBbXSwgInZhbF9tYWNyb19mMSI6IFtdfQogICAgYmVzdF9mMSwgYmVzdF9zdGF0ZSwgcGF0aWVuY2VfbGVmdCA9IC0xLjAsIE5vbmUsIHRyYWluX2NmZy5wYXRpZW5jZQoKICAgIGZvciBlcG9jaCBpbiByYW5nZSh0cmFpbl9jZmcuZXBvY2hzKToKICAgICAgICBtb2RlbC50cmFpbigpCiAgICAgICAgZXBvY2hfbG9zcywgbl9iYXRjaGVzID0gMC4wLCAwCiAgICAgICAgZm9yIHN0ZXAsIGJhdGNoIGluIGVudW1lcmF0ZSh0cmFpbl9sb2FkZXIpOgogICAgICAgICAgICBiYXRjaCA9IF9tb3ZlKGJhdGNoLCBkZXZpY2UpCiAgICAgICAgICAgIG9wdGltaXplci56ZXJvX2dyYWQoKQogICAgICAgICAgICBsb3NzLCBfID0gY29tcHV0ZV9sb3NzKG1vZGVsLCBiYXRjaCwgbW9kZWxfY2ZnKQogICAgICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICAgICAgdG9yY2gubm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKG1vZGVsLnBhcmFtZXRlcnMoKSwgdHJhaW5fY2ZnLm1heF9ncmFkX25vcm0pCiAgICAgICAgICAgIG9wdGltaXplci5zdGVwKCkKICAgICAgICAgICAgc2NoZWR1bGVyLnN0ZXAoKQogICAgICAgICAgICBlcG9jaF9sb3NzICs9IGZsb2F0KGxvc3MuaXRlbSgpKQogICAgICAgICAgICBuX2JhdGNoZXMgKz0gMQogICAgICAgICAgICBpZiB0cmFpbl9jZmcubG9nX2V2ZXJ5IGFuZCBzdGVwICUgdHJhaW5fY2ZnLmxvZ19ldmVyeSA9PSAwOgogICAgICAgICAgICAgICAgcHJpbnQoZiIgIGVwb2NoIHtlcG9jaCsxfSBzdGVwIHtzdGVwfS97bGVuKHRyYWluX2xvYWRlcil9IGxvc3Mge2xvc3MuaXRlbSgpOi40Zn0iKQogICAgICAgIGF2Z19sb3NzID0gZXBvY2hfbG9zcyAvIG1heCgxLCBuX2JhdGNoZXMpCiAgICAgICAgdmFsX2YxID0gX3ZhbF9tYWNyb19mMShtb2RlbCwgdmFsX2xvYWRlciwgZGV2aWNlKQogICAgICAgIGhpc3RvcnlbInRyYWluX2xvc3MiXS5hcHBlbmQoYXZnX2xvc3MpCiAgICAgICAgaGlzdG9yeVsidmFsX21hY3JvX2YxIl0uYXBwZW5kKHZhbF9mMSkKICAgICAgICBwcmludChmIltzZWVkIHtzZWVkfXx7bW9kZWxfY2ZnLnZhcmlhbnR9XSBlcG9jaCB7ZXBvY2grMX0ve3RyYWluX2NmZy5lcG9jaHN9ICIKICAgICAgICAgICAgICBmImxvc3M9e2F2Z19sb3NzOi40Zn0gdmFsX21hY3JvRjE9e3ZhbF9mMTouNGZ9IikKICAgICAgICBpZiB2YWxfZjEgPiBiZXN0X2YxOgogICAgICAgICAgICBiZXN0X2YxID0gdmFsX2YxCiAgICAgICAgICAgIGJlc3Rfc3RhdGUgPSBjb3B5LmRlZXBjb3B5KHtrOiB2LmNwdSgpIGZvciBrLCB2IGluIG1vZGVsLnN0YXRlX2RpY3QoKS5pdGVtcygpfSkKICAgICAgICAgICAgcGF0aWVuY2VfbGVmdCA9IHRyYWluX2NmZy5wYXRpZW5jZQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHBhdGllbmNlX2xlZnQgLT0gMQogICAgICAgICAgICBpZiBwYXRpZW5jZV9sZWZ0IDw9IDA6CiAgICAgICAgICAgICAgICBwcmludChmIltzZWVkIHtzZWVkfXx7bW9kZWxfY2ZnLnZhcmlhbnR9XSBlYXJseSBzdG9wIGF0IGVwb2NoIHtlcG9jaCsxfSIpCiAgICAgICAgICAgICAgICBicmVhawoKICAgIGlmIGJlc3Rfc3RhdGUgaXMgbm90IE5vbmU6CiAgICAgICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KGJlc3Rfc3RhdGUpCiAgICBoaXN0b3J5WyJiZXN0X3ZhbF9tYWNyb19mMSJdID0gYmVzdF9mMQogICAgcmV0dXJuIG1vZGVsLCBoaXN0b3J5Cg==",
  'src/faithdetect/models.py': "IiIiTW9kZWwgdmFyaWFudHMgZm9yIEZBSVRILURldGVjdC4KCkFsbCB0aHJlZSB2YXJpYW50cyBzaGFyZSBPTkUgYXJjaGl0ZWN0dXJlIChhIHRyYW5zZm9ybWVyIGVuY29kZXIgKyBsaW5lYXIgaGVhZCBvdmVyIHRoZQpbQ0xTXS88cz4gcmVwcmVzZW50YXRpb24pOyB0aGV5IGRpZmZlciBvbmx5IGluICpob3cgZGF0YSBpcyBmZWQqIGFuZCAqd2hhdCBsb3NzIGlzIHVzZWQqOgoKICAqIGBgYmFzZWxpbmVgYCA6IGZ1bGwgdGV4dCwgY3Jvc3MtZW50cm9weS4gICAtPiBzaG93cyBzaG9ydGN1dCBsZWFybmluZy4KICAqIGBgaGFyZG1hc2tgYCA6IGZ1bmN0aW9uLXdvcmQgdG9rZW4gaWRzIHJlcGxhY2VkIGJ5IHRoZSBgYFtGVU5DXWBgIHBsYWNlaG9sZGVyIChkb25lIGluCiAgICAgICAgICAgICAgICAgICB0aGUgQ29sbGF0b3IpLCBjcm9zcy1lbnRyb3B5LiBUaGUgZGVjaXNpb24gaXMgUFJPVkFCTFkgaW52YXJpYW50IHRvCiAgICAgICAgICAgICAgICAgICBmdW5jdGlvbi13b3JkIGlkZW50aXR5ICh0aGUgZW5jb2RlciBuZXZlciBzZWVzIHdoaWNoIGZ1bmN0aW9uIHdvcmQgaXQgd2FzKS4KICAqIGBgc29mdHJlZ2BgICA6IGZ1bGwgdGV4dCwgY3Jvc3MtZW50cm9weSArIGxhbWJkYSAqIChzYWxpZW5jeSBtYXNzIG9uIGZ1bmN0aW9uLXdvcmQKICAgICAgICAgICAgICAgICAgIHRva2VucykuIEEgInJpZ2h0LWZvci10aGUtcmlnaHQtcmVhc29ucyIgaW5wdXQtZ3JhZGllbnQgcGVuYWx0eSB0aGF0CiAgICAgICAgICAgICAgICAgICBkaXNjb3VyYWdlcywgYnV0IGRvZXMgbm90IGZvcmJpZCwgcmVsaWFuY2Ugb24gZnVuY3Rpb24gd29yZHMuCgpBIHNpbmdsZSBjbGVhbiBhcmNoaXRlY3R1cmUga2VlcHMgdGhlIHRocmVlIHZhcmlhbnRzIGRpcmVjdGx5IGNvbXBhcmFibGUuIFRoZSBgYFtGVU5DXWBgCnRva2VuIGlzIGFkZGVkIGZvciBldmVyeSB2YXJpYW50ICh1bnVzZWQgYnkgYmFzZWxpbmUvc29mdHJlZykKc28gYWxsIGNoZWNrcG9pbnRzIHNoYXJlIG9uZSB2b2NhYnVsYXJ5LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZAoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCmZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWwsIEF1dG9Ub2tlbml6ZXIKCmZyb20gLmZ1bmN0aW9uX3dvcmRzIGltcG9ydCBhZGRfZnVuY190b2tlbgoKVkFSSUFOVFMgPSAoImJhc2VsaW5lIiwgImhhcmRtYXNrIiwgInNvZnRyZWciLCAiZGVsZXRpb24iLCAicmFuZG9tIikKCgpAZGF0YWNsYXNzCmNsYXNzIE1vZGVsQ29uZmlnOgogICAgZW5jb2Rlcl9uYW1lOiBzdHIgPSAicm9iZXJ0YS1iYXNlIgogICAgdmFyaWFudDogc3RyID0gImJhc2VsaW5lIiAgICAgICAgICAjIG9uZSBvZiBWQVJJQU5UUwogICAgbnVtX2xhYmVsczogaW50ID0gMgogICAgZHJvcG91dDogZmxvYXQgPSAwLjEKICAgIG1heF9sZW5ndGg6IGludCA9IDI1NgogICAgc29mdHJlZ19sYW1iZGE6IGZsb2F0ID0gMS4wICAgICAgICAjIHdlaWdodCBvZiB0aGUgYXR0cmlidXRpb24gcGVuYWx0eSAoc29mdHJlZyBvbmx5KQogICAgc29mdHJlZ19wZW5hbHR5X2JhdGNoOiBpbnQgPSA4ICAgICAjIGNhcCBleGFtcGxlcyBpbiB0aGUgMm5kLW9yZGVyIHBlbmFsdHkgKGJvdW5kcyBtZW1vcnkpCiAgICBwb29saW5nOiBzdHIgPSAiY2xzIiAgICAgICAgICAgICAgICMgImNscyIgb3IgIm1lYW4iCgogICAgZGVmIF9fcG9zdF9pbml0X18oc2VsZik6CiAgICAgICAgaWYgc2VsZi52YXJpYW50IG5vdCBpbiBWQVJJQU5UUzoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInZhcmlhbnQgbXVzdCBiZSBvbmUgb2Yge1ZBUklBTlRTfSwgZ290IHtzZWxmLnZhcmlhbnQhcn0iKQoKCmRlZiBidWlsZF90b2tlbml6ZXIoZW5jb2Rlcl9uYW1lOiBzdHIgPSAicm9iZXJ0YS1iYXNlIik6CiAgICAiIiJSZXR1cm4gKHRva2VuaXplciwgZnVuY190b2tlbl9pZCkgd2l0aCB0aGUgYGBbRlVOQ11gYCBwbGFjZWhvbGRlciByZWdpc3RlcmVkLiIiIgogICAgdG9rID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoZW5jb2Rlcl9uYW1lLCB1c2VfZmFzdD1UcnVlKQogICAgZnVuY19pZCA9IGFkZF9mdW5jX3Rva2VuKHRvaykKICAgIHJldHVybiB0b2ssIGZ1bmNfaWQKCgpjbGFzcyBSZXZpZXdEZXRlY3Rvcihubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvbmZpZzogTW9kZWxDb25maWcsIHZvY2FiX3NpemU6IGludCB8IE5vbmUgPSBOb25lKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbmZpZyA9IGNvbmZpZwogICAgICAgICMgRm9yY2UgZWFnZXIgYXR0ZW50aW9uOiB0aGUgZnVzZWQgU0RQQSBrZXJuZWwgbGFja3MgYSBDUFUgZG91YmxlLWJhY2t3YXJkIChuZWVkZWQgYnkKICAgICAgICAjIHRoZSBzb2Z0LXJlZyBhdHRyaWJ1dGlvbiBwZW5hbHR5KSBhbmQgYnVzLWVycm9ycyB1bmRlciBDYXB0dW0gSUcgb24gQ1BVLiBFYWdlcgogICAgICAgICMgYXR0ZW50aW9uIGlzIGNvcnJlY3Qgb24gQ1BVL01QUy9DVURBIGFuZCBvbmx5IG1hcmdpbmFsbHkgc2xvd2VyIGZvciB0aGVzZSBzaXplcy4KICAgICAgICB0cnk6CiAgICAgICAgICAgIHNlbGYuZW5jb2RlciA9IEF1dG9Nb2RlbC5mcm9tX3ByZXRyYWluZWQoY29uZmlnLmVuY29kZXJfbmFtZSwgYXR0bl9pbXBsZW1lbnRhdGlvbj0iZWFnZXIiKQogICAgICAgIGV4Y2VwdCAoVHlwZUVycm9yLCBWYWx1ZUVycm9yKToKICAgICAgICAgICAgc2VsZi5lbmNvZGVyID0gQXV0b01vZGVsLmZyb21fcHJldHJhaW5lZChjb25maWcuZW5jb2Rlcl9uYW1lKQogICAgICAgIGlmIHZvY2FiX3NpemUgaXMgbm90IE5vbmUgYW5kIHZvY2FiX3NpemUgIT0gc2VsZi5lbmNvZGVyLmdldF9pbnB1dF9lbWJlZGRpbmdzKCkud2VpZ2h0LnNoYXBlWzBdOgogICAgICAgICAgICBzZWxmLl9yZXNpemVfZW1iZWRkaW5ncyh2b2NhYl9zaXplKQogICAgICAgIGhpZGRlbiA9IHNlbGYuZW5jb2Rlci5jb25maWcuaGlkZGVuX3NpemUKICAgICAgICBzZWxmLmRyb3BvdXQgPSBubi5Ecm9wb3V0KGNvbmZpZy5kcm9wb3V0KQogICAgICAgIHNlbGYuY2xhc3NpZmllciA9IG5uLkxpbmVhcihoaWRkZW4sIGNvbmZpZy5udW1fbGFiZWxzKQoKICAgIGRlZiBfcmVzaXplX2VtYmVkZGluZ3Moc2VsZiwgdm9jYWJfc2l6ZTogaW50KSAtPiBOb25lOgogICAgICAgIG9sZCA9IHNlbGYuZW5jb2Rlci5nZXRfaW5wdXRfZW1iZWRkaW5ncygpLndlaWdodC5kYXRhCiAgICAgICAgc2VsZi5lbmNvZGVyLnJlc2l6ZV90b2tlbl9lbWJlZGRpbmdzKHZvY2FiX3NpemUpCiAgICAgICAgIyBJbml0aWFsaXNlIGFueSBuZXcgcm93cyAoZS5nLiBbRlVOQ10pIHRvIHRoZSBtZWFuIG9mIGV4aXN0aW5nIGVtYmVkZGluZ3MgZm9yIHN0YWJpbGl0eS4KICAgICAgICBuZXcgPSBzZWxmLmVuY29kZXIuZ2V0X2lucHV0X2VtYmVkZGluZ3MoKS53ZWlnaHQuZGF0YQogICAgICAgIGlmIHZvY2FiX3NpemUgPiBvbGQuc2hhcGVbMF06CiAgICAgICAgICAgIG5ld1tvbGQuc2hhcGVbMF06XSA9IG9sZC5tZWFuKGRpbT0wLCBrZWVwZGltPVRydWUpCgogICAgZGVmIHdvcmRfZW1iZWRkaW5ncyhzZWxmLCBpbnB1dF9pZHM6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHJldHVybiBzZWxmLmVuY29kZXIuZ2V0X2lucHV0X2VtYmVkZGluZ3MoKShpbnB1dF9pZHMpCgogICAgZGVmIF9wb29sKHNlbGYsIGxhc3RfaGlkZGVuX3N0YXRlOiB0b3JjaC5UZW5zb3IsIGF0dGVudGlvbl9tYXNrOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBpZiBzZWxmLmNvbmZpZy5wb29saW5nID09ICJjbHMiOgogICAgICAgICAgICByZXR1cm4gbGFzdF9oaWRkZW5fc3RhdGVbOiwgMF0KICAgICAgICBtYXNrID0gYXR0ZW50aW9uX21hc2sudW5zcXVlZXplKC0xKS5mbG9hdCgpCiAgICAgICAgc3VtbWVkID0gKGxhc3RfaGlkZGVuX3N0YXRlICogbWFzaykuc3VtKDEpCiAgICAgICAgY291bnRzID0gbWFzay5zdW0oMSkuY2xhbXBfbWluKDFlLTkpCiAgICAgICAgcmV0dXJuIHN1bW1lZCAvIGNvdW50cwoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGlucHV0X2lkcz1Ob25lLCBhdHRlbnRpb25fbWFzaz1Ob25lLCBpbnB1dHNfZW1iZWRzPU5vbmUpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBrd2FyZ3MgPSB7ImF0dGVudGlvbl9tYXNrIjogYXR0ZW50aW9uX21hc2t9CiAgICAgICAgaWYgaW5wdXRzX2VtYmVkcyBpcyBub3QgTm9uZToKICAgICAgICAgICAga3dhcmdzWyJpbnB1dHNfZW1iZWRzIl0gPSBpbnB1dHNfZW1iZWRzCiAgICAgICAgZWxzZToKICAgICAgICAgICAga3dhcmdzWyJpbnB1dF9pZHMiXSA9IGlucHV0X2lkcwogICAgICAgIG91dCA9IHNlbGYuZW5jb2RlcigqKmt3YXJncykKICAgICAgICBwb29sZWQgPSBzZWxmLl9wb29sKG91dC5sYXN0X2hpZGRlbl9zdGF0ZSwgYXR0ZW50aW9uX21hc2spCiAgICAgICAgcmV0dXJuIHNlbGYuY2xhc3NpZmllcihzZWxmLmRyb3BvdXQocG9vbGVkKSkKCgpkZWYgc29mdHJlZ19wZW5hbHR5KAogICAgbW9kZWw6IFJldmlld0RldGVjdG9yLAogICAgaW5wdXRfaWRzOiB0b3JjaC5UZW5zb3IsCiAgICBhdHRlbnRpb25fbWFzazogdG9yY2guVGVuc29yLAogICAgZndfbWFzazogdG9yY2guVGVuc29yLAogICAgbGFiZWxzOiB0b3JjaC5UZW5zb3IsCiAgICBtYXhfYmF0Y2g6IGludCA9IDgsCikgLT4gdG9yY2guVGVuc29yOgogICAgIiIiU2FsaWVuY3ktb24tZnVuY3Rpb24td29yZHMgcGVuYWx0eSwgY29tcHV0ZWQgb24gKHVwIHRvKSBgYG1heF9iYXRjaGBgIGV4YW1wbGVzLgoKICAgIHBlbmFsdHkgPSBtZWFuIG9mIChzYWxpZW5jeSBtYXNzIG9uIGZ1bmN0aW9uLXdvcmQgdG9rZW5zIC8gdG90YWwgc2FsaWVuY3kpLCB3aGVyZSBwZXItdG9rZW4KICAgIHNhbGllbmN5ID0gfGdyYWRpZW50KHRydWUtY2xhc3MgbG9naXQpIC4gaW5wdXQgZW1iZWRkaW5nfCAoZ3JhZGllbnQgeCBpbnB1dCkuIGBgY3JlYXRlX2dyYXBoCiAgICA9VHJ1ZWBgIGxldHMgdGhlIHBlbmFsdHkgYmFja3Byb3AgaW50byBtb2RlbCB3ZWlnaHRzIChSb3NzIGV0IGFsLiwgMjAxNywgIlJpZ2h0IGZvciB0aGUgUmlnaHQKICAgIFJlYXNvbnMiKS4gVGhlIHNlY29uZC1vcmRlciBncmFwaCBpcyB0aGUgbWVtb3J5IGJvdHRsZW5lY2ssIHNvIHdlIGNhcCBpdCB0byBhIHNtYWxsCiAgICBzdWItYmF0Y2gg4oCUIGFuIHVuYmlhc2VkIHN0b2NoYXN0aWMgZXN0aW1hdGUgb2YgdGhlIHJlZ3VsYXJpc2VyIHRoYXQga2VlcHMgbWVtb3J5IGJvdW5kZWQKICAgIChpbXBvcnRhbnQgb24gYSBzaW5nbGUgR1BVIGF0IGxhcmdlIGJhdGNoL3NlcSkuCiAgICAiIiIKICAgIGsgPSBtaW4oaW50KG1heF9iYXRjaCksIGlucHV0X2lkcy5zaGFwZVswXSkKICAgIGlpLCBhbSwgZm0sIGxiID0gaW5wdXRfaWRzWzprXSwgYXR0ZW50aW9uX21hc2tbOmtdLCBmd19tYXNrWzprXSwgbGFiZWxzWzprXQogICAgZW1iID0gbW9kZWwud29yZF9lbWJlZGRpbmdzKGlpKS5kZXRhY2goKS5jbG9uZSgpLnJlcXVpcmVzX2dyYWRfKFRydWUpCiAgICBsb2dpdHMgPSBtb2RlbChpbnB1dHNfZW1iZWRzPWVtYiwgYXR0ZW50aW9uX21hc2s9YW0pCiAgICB0YXJnZXQgPSBsb2dpdHMuZ2F0aGVyKDEsIGxiLnZpZXcoLTEsIDEpKS5zdW0oKQogICAgZ3JhZHMgPSB0b3JjaC5hdXRvZ3JhZC5ncmFkKHRhcmdldCwgZW1iLCBjcmVhdGVfZ3JhcGg9VHJ1ZSlbMF0gICAjIFtrLCBMLCBIXQogICAgdG9rZW5fc2FsaWVuY3kgPSAoZ3JhZHMgKiBlbWIpLnN1bSgtMSkuYWJzKCkgICAgICAgICAgICAgICAgICAgICAjIFtrLCBMXQogICAgcmVhbCA9IGFtLmZsb2F0KCkKICAgIGZ3ID0gZm0uZmxvYXQoKSAqIHJlYWwKICAgIG51bSA9ICh0b2tlbl9zYWxpZW5jeSAqIGZ3KS5zdW0oMSkKICAgIGRlbiA9ICh0b2tlbl9zYWxpZW5jeSAqIHJlYWwpLnN1bSgxKS5jbGFtcF9taW4oMWUtOCkKICAgIHJldHVybiAobnVtIC8gZGVuKS5tZWFuKCkKCgojIEJhY2t3YXJkcy1jb21wYXRpYmxlIGFsaWFzIChmdWxsLWJhdGNoIHBlbmFsdHkgKyBsb2dpdHMpIHVzZWQgYnkgc29tZSB0ZXN0cy4KZGVmIHNhbGllbmN5X29uX2Z1bmN0aW9uX3dvcmRzKG1vZGVsLCBpbnB1dF9pZHMsIGF0dGVudGlvbl9tYXNrLCBmd19tYXNrLCBsYWJlbHMpOgogICAgZW1iID0gbW9kZWwud29yZF9lbWJlZGRpbmdzKGlucHV0X2lkcykuZGV0YWNoKCkuY2xvbmUoKS5yZXF1aXJlc19ncmFkXyhUcnVlKQogICAgbG9naXRzID0gbW9kZWwoaW5wdXRzX2VtYmVkcz1lbWIsIGF0dGVudGlvbl9tYXNrPWF0dGVudGlvbl9tYXNrKQogICAgdGFyZ2V0ID0gbG9naXRzLmdhdGhlcigxLCBsYWJlbHMudmlldygtMSwgMSkpLnN1bSgpCiAgICBncmFkcyA9IHRvcmNoLmF1dG9ncmFkLmdyYWQodGFyZ2V0LCBlbWIsIGNyZWF0ZV9ncmFwaD1UcnVlKVswXQogICAgdG9rZW5fc2FsaWVuY3kgPSAoZ3JhZHMgKiBlbWIpLnN1bSgtMSkuYWJzKCkKICAgIHJlYWwgPSBhdHRlbnRpb25fbWFzay5mbG9hdCgpCiAgICBmdyA9IGZ3X21hc2suZmxvYXQoKSAqIHJlYWwKICAgIG51bSA9ICh0b2tlbl9zYWxpZW5jeSAqIGZ3KS5zdW0oMSkKICAgIGRlbiA9ICh0b2tlbl9zYWxpZW5jeSAqIHJlYWwpLnN1bSgxKS5jbGFtcF9taW4oMWUtOCkKICAgIHJldHVybiBsb2dpdHMsIChudW0gLyBkZW4pLm1lYW4oKQoKCmRlZiBjb21wdXRlX2xvc3MoCiAgICBtb2RlbDogUmV2aWV3RGV0ZWN0b3IsCiAgICBiYXRjaDogZGljdCwKICAgIGNvbmZpZzogTW9kZWxDb25maWcsCikgLT4gdHVwbGVbdG9yY2guVGVuc29yLCB0b3JjaC5UZW5zb3JdOgogICAgIiIiUmV0dXJuIChsb3NzLCBsb2dpdHMpIGZvciB0aGUgY29uZmlndXJlZCB2YXJpYW50LgoKICAgIENyb3NzLWVudHJvcHkgYWx3YXlzIHVzZXMgdGhlIGZ1bGwgYmF0Y2ggKGEgbm9ybWFsIGZvcndhcmQpLiBGb3Igc29mdC1yZWcgdGhlIGF0dHJpYnV0aW9uCiAgICBwZW5hbHR5IGlzIGFkZGVkIGZyb20gYSBzbWFsbCBzdWItYmF0Y2ggc2Vjb25kLW9yZGVyIHBhc3MsIHNvIHBlYWsgbWVtb3J5IHN0YXlzIGNsb3NlIHRvCiAgICBvcmRpbmFyeSBmaW5lLXR1bmluZyByZWdhcmRsZXNzIG9mIGJhdGNoIHNpemUgLyBzZXF1ZW5jZSBsZW5ndGguCiAgICAiIiIKICAgIGxhYmVscyA9IGJhdGNoWyJsYWJlbHMiXQogICAgbG9naXRzID0gbW9kZWwoaW5wdXRfaWRzPWJhdGNoWyJpbnB1dF9pZHMiXSwgYXR0ZW50aW9uX21hc2s9YmF0Y2hbImF0dGVudGlvbl9tYXNrIl0pCiAgICBsb3NzID0gRi5jcm9zc19lbnRyb3B5KGxvZ2l0cywgbGFiZWxzKQogICAgaWYgY29uZmlnLnZhcmlhbnQgPT0gInNvZnRyZWciOgogICAgICAgIHBlbmFsdHkgPSBzb2Z0cmVnX3BlbmFsdHkoCiAgICAgICAgICAgIG1vZGVsLCBiYXRjaFsiaW5wdXRfaWRzIl0sIGJhdGNoWyJhdHRlbnRpb25fbWFzayJdLCBiYXRjaFsiZndfbWFzayJdLCBsYWJlbHMsCiAgICAgICAgICAgIG1heF9iYXRjaD1jb25maWcuc29mdHJlZ19wZW5hbHR5X2JhdGNoLAogICAgICAgICkKICAgICAgICBsb3NzID0gbG9zcyArIGNvbmZpZy5zb2Z0cmVnX2xhbWJkYSAqIHBlbmFsdHkKICAgIHJldHVybiBsb3NzLCBsb2dpdHMK",
  'scripts/run_desirable.py': "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJEZXNpcmFibGUgcmV2aWV3ZXIgcmV2aXNpb25zLCB0cmFpbmVkIGF0IFBJTE9UIHNjYWxlIChkaXN0aWxyb2JlcnRhLWJhc2UsIDYwMC1zdWJzYW1wbGUpLgoKQS4gTWFza2luZy12YXJpYW50IGNvbXBhcmlzb24gLT4gcmVzdWx0cy92YXJpYW50X2NvbXBhcmlzb24uanNvbgogICBiYXNlbGluZSAvIGhhcmRtYXNrKFtGVU5DXSkgLyBkZWxldGlvbihyZW1vdmUgRlcpIC8gcmFuZG9tKHJhbmRvbS10b2tlbiBwbGFjZWhvbGRlciksCiAgIHRvIHNlcGFyYXRlIGZ1bmN0aW9uLXdvcmQgSURFTlRJVFksIHRoZSBzbG90LU1BUktFUiwgYW5kIFBPU0lUSU9OL0NPVU5UIGVmZmVjdHMuCiAgIChTb2Z0UmVnIGlzIGluIHRoZSBtYWluIHJ1bjsgY2F0ZWdvcnktc3BlY2lmaWMgbWFza3MgYXJlIGluIGNhdGVnb3J5X2FibGF0aW9uLmpzb24uKQoKQi4gU29mdFJlZyBsYW1iZGEgc3dlZXAgLT4gcmVzdWx0cy9sYW1iZGFfc3dlZXAuanNvbgogICBpbi1kb21haW4gRjEgYW5kIGlkZW50aXR5LXNlbnNpdGl2aXR5IGFjcm9zcyBsYW1iZGEsIHRyYWNpbmcgdGhlIHJlbGlhbmNlLS1hY2N1cmFjeSBmcm9udGllci4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKaW1wb3J0IGFyZ3BhcnNlLCBnYywganNvbiwgbWF0aCwgb3MsIHN5cywgdGltZQppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKCnN5cy5wYXRoLmluc2VydCgwLCAic3JjIikKZnJvbSBmYWl0aGRldGVjdC5tb2RlbHMgaW1wb3J0IFJldmlld0RldGVjdG9yLCBNb2RlbENvbmZpZywgYnVpbGRfdG9rZW5pemVyICAjIG5vcWEKZnJvbSBmYWl0aGRldGVjdC5mdW5jdGlvbl93b3JkcyBpbXBvcnQgYnVpbGRfZnVuY3Rpb25fd29yZF9zZXQKZnJvbSBmYWl0aGRldGVjdC5kYXRhIGltcG9ydCBsb2FkX21haWRlX3VwX2VuZ2xpc2gsIG1ha2Vfc3BsaXRzCmZyb20gZmFpdGhkZXRlY3QudHJhaW4gaW1wb3J0IFRyYWluQ29uZmlnLCB0cmFpbl9tb2RlbApmcm9tIGZhaXRoZGV0ZWN0LmV2YWx1YXRlIGltcG9ydCBldmFsdWF0ZV9zcGxpdApmcm9tIGZhaXRoZGV0ZWN0LmV4cGVyaW1lbnQgaW1wb3J0IF9zdWJzYW1wbGUKZnJvbSBmYWl0aGRldGVjdC5leHBsYWluLmF0dHJpYnV0aW9ucyBpbXBvcnQgRmFpdGhmdWxFeHBsYWluZXIKZnJvbSBmYWl0aGRldGVjdC5leHBsYWluLmZhaXRoZnVsbmVzcyBpbXBvcnQgZnVuY3Rpb25fd29yZF9pZGVudGl0eV9zZW5zaXRpdml0eQoKIyBEZWZhdWx0cyBhcmUgdGhlIGxvY2FsIFBJTE9UIGNvbmZpZzsgdGhlIENvbGFiIG5vdGVib29rIG92ZXJyaWRlcyB0aGVtIHZpYSBDTEkgZm9yIHRoZSBmdWxsIHJ1bi4KREFUQSwgRU5DLCBNQVhMRU4sIEZXREVGLCBTVUIsIEVQT0NIUyA9ICJkYXRhL2FsbF9kYXRhLmNzdiIsICJkaXN0aWxyb2JlcnRhLWJhc2UiLCAxMTIsICJ1bmlvbiIsIDYwMCwgMwpPT0QgPSAicmVzdWx0cy9jYWNoZS9yYWlkX29vZC5wYXJxdWV0IgpERVZJQ0VfT1ZFUlJJREUgPSBOb25lICAgICAgIyBzZXQgZnJvbSAtLWRldmljZQpQRU5BTFRZX0JBVENIID0gMiAgICAgICAgICAgIyBzb2Z0cmVnIDJuZC1vcmRlciBzdWItYmF0Y2ggKHJhaXNlIG9uIEdQVSkKCgpkZWYgZGV2KCk6CiAgICBpZiBERVZJQ0VfT1ZFUlJJREU6CiAgICAgICAgcmV0dXJuIHRvcmNoLmRldmljZShERVZJQ0VfT1ZFUlJJREUpCiAgICBmb3JjZWQgPSBvcy5lbnZpcm9uLmdldCgiRkRfREVWSUNFIikgICAjIEZEX0RFVklDRT1jcHUgYXZvaWRzIE1QUyBPT00gb24gdGhlIDggR0IgYm94CiAgICBpZiBmb3JjZWQ6CiAgICAgICAgcmV0dXJuIHRvcmNoLmRldmljZShmb3JjZWQpCiAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJldHVybiB0b3JjaC5kZXZpY2UoImN1ZGEiKQogICAgcmV0dXJuIHRvcmNoLmRldmljZSgibXBzIikgaWYgdG9yY2guYmFja2VuZHMubXBzLmlzX2F2YWlsYWJsZSgpIGVsc2UgdG9yY2guZGV2aWNlKCJjcHUiKQoKCmRlZiBfY2xlYW51cCgpOgogICAgZ2MuY29sbGVjdCgpCiAgICBpZiB0b3JjaC5iYWNrZW5kcy5tcHMuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICB0b3JjaC5tcHMuZW1wdHlfY2FjaGUoKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKCgpkZWYgY2kodmFscyk6CiAgICBhID0gbnAuYXJyYXkodmFscywgZHR5cGU9ZmxvYXQpOyBuID0gbGVuKGEpOyBtID0gZmxvYXQoYS5tZWFuKCkpCiAgICBpZiBuIDwgMjoKICAgICAgICByZXR1cm4geyJtZWFuIjogbSwgImxvIjogbSwgImhpIjogbSwgIm4iOiBufQogICAgc2QgPSBmbG9hdChhLnN0ZChkZG9mPTEpKTsgdCA9IDEyLjcwNiBpZiBuID09IDIgZWxzZSAoNC4zMDMgaWYgbiA9PSAzIGVsc2UgMi43NzYpCiAgICBoID0gdCAqIHNkIC8gbWF0aC5zcXJ0KG4pCiAgICByZXR1cm4geyJtZWFuIjogbSwgImxvIjogbSAtIGgsICJoaSI6IG0gKyBoLCAic2QiOiBzZCwgIm4iOiBufQoKCmRlZiBfcHJlcChzZWVkKToKICAgIGZ3ID0gYnVpbGRfZnVuY3Rpb25fd29yZF9zZXQoRldERUYpCiAgICB0b2ssIGZpZCA9IGJ1aWxkX3Rva2VuaXplcihFTkMpCiAgICBkZiA9IGxvYWRfbWFpZGVfdXBfZW5nbGlzaChEQVRBKQogICAgc3AgPSBtYWtlX3NwbGl0cyhkZiwgbW9kZT0iZ3JvdXBlZCIsIHNlZWQ9c2VlZCkKICAgIGlmIFNVQiBhbmQgU1VCID4gMDogICAgICAgICAgICAgICAgICMgU1VCPTAvTm9uZSAtPiBmdWxsIHRyYWluaW5nIHNldCAoQ29sYWIgZnVsbCBzY2FsZSkKICAgICAgICBzcC50cmFpbiA9IF9zdWJzYW1wbGUoc3AudHJhaW4sIFNVQiwgc2VlZCkKICAgIHJldHVybiBmdywgdG9rLCBmaWQsIHNwCgoKZGVmIF90cmFpbl9ldmFsKHZhcmlhbnQsIHNlZWQsIGxhbT0xLjAsIG5lZWRfb29kPVRydWUpOgogICAgZGV2aWNlID0gZGV2KCkKICAgIGZ3LCB0b2ssIGZpZCwgc3AgPSBfcHJlcChzZWVkKQogICAgIyBzb2Z0cmVnX3BlbmFsdHlfYmF0Y2ggYm91bmRzIHRoZSBzZWNvbmQtb3JkZXIgKGNyZWF0ZV9ncmFwaCkgcGVuYWx0eSBtZW1vcnkgKHNtYWxsIG9uIHRoZQogICAgIyA4IEdCIGJveDsgdGhlIENvbGFiIG5vdGVib29rIHJhaXNlcyBpdCBvbiBHUFUpLiBiYXRjaF9zaXplIHNocmlua3MgZm9yIHNvZnRyZWcgdG9vLgogICAgbWNmZyA9IE1vZGVsQ29uZmlnKGVuY29kZXJfbmFtZT1FTkMsIHZhcmlhbnQ9dmFyaWFudCwgbWF4X2xlbmd0aD1NQVhMRU4sCiAgICAgICAgICAgICAgICAgICAgICAgc29mdHJlZ19sYW1iZGE9bGFtLCBzb2Z0cmVnX3BlbmFsdHlfYmF0Y2g9UEVOQUxUWV9CQVRDSCkKICAgIGJzID0gOCBpZiB2YXJpYW50ID09ICJzb2Z0cmVnIiBlbHNlIDE2CiAgICB0Y2ZnID0gVHJhaW5Db25maWcoZXBvY2hzPUVQT0NIUywgYmF0Y2hfc2l6ZT1icywgbHI9MmUtNSkKICAgIG1vZGVsLCBoaXN0ID0gdHJhaW5fbW9kZWwobWNmZywgdGNmZywgc3AsIHRvaywgZmlkLCBmdywgZGV2aWNlLCBzZWVkKQogICAgaW5kID0gZXZhbHVhdGVfc3BsaXQobW9kZWwsIHNwLnRlc3QsIG1jZmcsIHRvaywgZmlkLCBmdywgZGV2aWNlKVsibWV0cmljcyJdCiAgICBvb2QgPSBOb25lCiAgICBpZiBuZWVkX29vZCBhbmQgT09EIGFuZCBvcy5wYXRoLmV4aXN0cyhPT0QpOgogICAgICAgIG9vZF9kZiA9IHBkLnJlYWRfcGFycXVldChPT0QpW1sidGV4dCIsICJsYWJlbCJdXQogICAgICAgIG9vZCA9IGV2YWx1YXRlX3NwbGl0KG1vZGVsLCBvb2RfZGYsIG1jZmcsIHRvaywgZmlkLCBmdywgZGV2aWNlKVsibWV0cmljcyJdCiAgICByZXR1cm4gbW9kZWwsIG1jZmcsIHRvaywgZmlkLCBmdywgZGV2aWNlLCBzcCwgaW5kLCBvb2QKCgpkZWYgcnVuX3ZhcmlhbnRzKHNlZWRzKToKICAgIHZhcmlhbnRzID0gWyJiYXNlbGluZSIsICJoYXJkbWFzayIsICJkZWxldGlvbiIsICJyYW5kb20iXQogICAgcGVyID0ge3Y6IHsiaW5kb21haW5fZjEiOiBbXSwgIm9vZF9mMSI6IFtdfSBmb3IgdiBpbiB2YXJpYW50c30KICAgIGZvciB2IGluIHZhcmlhbnRzOgogICAgICAgIGZvciBzIGluIHNlZWRzOgogICAgICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgICAgIHJlcyA9IF90cmFpbl9ldmFsKHYsIHMpCiAgICAgICAgICAgIG1vZGVsLCBpbmQsIG9vZCA9IHJlc1swXSwgcmVzWzddLCByZXNbOF0KICAgICAgICAgICAgb29kZjEgPSBvb2RbImYxX21hY3JvIl0gKiAxMDAgaWYgb29kIGVsc2UgZmxvYXQoIm5hbiIpCiAgICAgICAgICAgIHBlclt2XVsiaW5kb21haW5fZjEiXS5hcHBlbmQoaW5kWyJmMV9tYWNybyJdICogMTAwKQogICAgICAgICAgICBwZXJbdl1bIm9vZF9mMSJdLmFwcGVuZChvb2RmMSkKICAgICAgICAgICAgcHJpbnQoZiJbdmFyXSB7dn0gc2VlZHtzfTogaW49e2luZFsnZjFfbWFjcm8nXSoxMDA6LjFmfSBvb2Q9e29vZGYxOi4xZn0gKHt0aW1lLnRpbWUoKS10MDouMGZ9cykiLCBmbHVzaD1UcnVlKQogICAgICAgICAgICBkZWwgbW9kZWwsIHJlcwogICAgICAgICAgICBfY2xlYW51cCgpCiAgICAgICAgIyBpbmNyZW1lbnRhbCB3cml0ZSBzbyBhIGNyYXNoIG1pZC1ydW4gaXMgcmVjb3ZlcmFibGUKICAgICAgICBqc29uLmR1bXAoeyJ2YXJpYW50cyI6IHt2djogeyJpbmRvbWFpbl9mMSI6IGNpKHBlclt2dl1bImluZG9tYWluX2YxIl0pLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIm9vZF9mMSI6IGNpKHBlclt2dl1bIm9vZF9mMSJdKX0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgdnYgaW4gdmFyaWFudHMgaWYgcGVyW3Z2XVsiaW5kb21haW5fZjEiXX19LAogICAgICAgICAgICAgICAgICBvcGVuKCJyZXN1bHRzL3ZhcmlhbnRfY29tcGFyaXNvbl9wYXJ0aWFsLmpzb24iLCAidyIpLCBpbmRlbnQ9MikKICAgIG91dCA9IHsibWV0YSI6IHsiZW5jb2RlciI6IEVOQywgInRyYWluX3N1YnNhbXBsZSI6IFNVQiwgImVwb2NocyI6IEVQT0NIUywgInNlZWRzIjogc2VlZHMsCiAgICAgICAgICAgICAgICAgICAgIm9vZCI6IE9PRCwKICAgICAgICAgICAgICAgICAgICAibm90ZSI6ICJiYXNlbGluZT1pZGVudGl0eStzbG90cytjb250ZW50OyBoYXJkbWFzaz1pZGVudGl0eSByZW1vdmVkIHNsb3Qga2VwdDsgIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgImRlbGV0aW9uPWlkZW50aXR5K3Nsb3QgcmVtb3ZlZDsgcmFuZG9tPWlkZW50aXR5IHJlbW92ZWQsIGNvbnNpc3RlbnQgc2xvdC1tYXJrZXIgcmVtb3ZlZC4ifSwKICAgICAgICAgICAidmFyaWFudHMiOiB7djogeyJpbmRvbWFpbl9mMSI6IGNpKHBlclt2XVsiaW5kb21haW5fZjEiXSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAib29kX2YxIjogY2kocGVyW3ZdWyJvb2RfZjEiXSl9IGZvciB2IGluIHZhcmlhbnRzfX0KICAgIGpzb24uZHVtcChvdXQsIG9wZW4oInJlc3VsdHMvdmFyaWFudF9jb21wYXJpc29uLmpzb24iLCAidyIpLCBpbmRlbnQ9MikKICAgIHByaW50KCJ3cm90ZSByZXN1bHRzL3ZhcmlhbnRfY29tcGFyaXNvbi5qc29uIiwgZmx1c2g9VHJ1ZSkKCgpkZWYgcnVuX2xhbWJkYShsYW1iZGFzLCBzZWVkcywgbl9zZW5zPTQwKToKICAgIHJvd3MgPSBbXQogICAgZm9yIGxhbSBpbiBsYW1iZGFzOgogICAgICAgIGYxcywgc2VucyA9IFtdLCBbXQogICAgICAgIGZvciBzIGluIHNlZWRzOgogICAgICAgICAgICBtb2RlbCwgbWNmZywgdG9rLCBmaWQsIGZ3LCBkZXZpY2UsIHNwLCBpbmQsIF8gPSBfdHJhaW5fZXZhbCgic29mdHJlZyIsIHMsIGxhbT1sYW0sIG5lZWRfb29kPUZhbHNlKQogICAgICAgICAgICBmMXMuYXBwZW5kKGluZFsiZjFfbWFjcm8iXSAqIDEwMCkKICAgICAgICAgICAgZXhwbCA9IEZhaXRoZnVsRXhwbGFpbmVyKG1vZGVsLCBtY2ZnLCB0b2ssIGZpZCwgZncsIGRldmljZSkKICAgICAgICAgICAgdGV4dHMgPSBsaXN0KHNwLnRlc3RbInRleHQiXS5oZWFkKG5fc2VucykpCiAgICAgICAgICAgIHZhbHMgPSBmdW5jdGlvbl93b3JkX2lkZW50aXR5X3NlbnNpdGl2aXR5KGV4cGwsIHRleHRzLCBzZWVkPXMsIG1heF90ZXh0cz1uX3NlbnMpCiAgICAgICAgICAgIHNlbnMuYXBwZW5kKGZsb2F0KG5wLm1lYW4oW2Ficyh4KSBmb3IgeCBpbiB2YWxzXSkpICogMTAwKQogICAgICAgICAgICBkZWwgbW9kZWwsIGV4cGwKICAgICAgICAgICAgX2NsZWFudXAoKQogICAgICAgIHJvd3MuYXBwZW5kKHsibGFtYmRhIjogbGFtLCAiaW5kb21haW5fZjEiOiBjaShmMXMpLCAiaWRlbnRpdHlfc2Vuc2l0aXZpdHkiOiBjaShzZW5zKX0pCiAgICAgICAgcHJpbnQoZiJbbGFtXSDOuz17bGFtfTogRjE9e25wLm1lYW4oZjFzKTouMWZ9IGlkZW50aXR5LXNlbnM9e25wLm1lYW4oc2Vucyk6LjJmfSUiLCBmbHVzaD1UcnVlKQogICAgb3V0ID0geyJtZXRhIjogeyJlbmNvZGVyIjogRU5DLCAidHJhaW5fc3Vic2FtcGxlIjogU1VCLCAiZXBvY2hzIjogRVBPQ0hTLCAic2VlZHMiOiBzZWVkcywKICAgICAgICAgICAgICAgICAgICAibm90ZSI6ICJTb2Z0UmVnIGxhbWJkYSBzd2VlcDsgbGFtYmRhPTAgfiBiYXNlbGluZS4ifSwgInN3ZWVwIjogcm93c30KICAgIGpzb24uZHVtcChvdXQsIG9wZW4oInJlc3VsdHMvbGFtYmRhX3N3ZWVwLmpzb24iLCAidyIpLCBpbmRlbnQ9MikKICAgIHByaW50KCJ3cm90ZSByZXN1bHRzL2xhbWJkYV9zd2VlcC5qc29uIiwgZmx1c2g9VHJ1ZSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbW9kZSIsIGNob2ljZXM9WyJ2YXJpYW50cyIsICJsYW1iZGEiLCAiYm90aCJdLCBkZWZhdWx0PSJib3RoIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zZWVkcyIsIHR5cGU9aW50LCBuYXJncz0iKyIsIGRlZmF1bHQ9WzAsIDEsIDJdKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWxhbWJkYS1zZWVkcyIsIHR5cGU9aW50LCBuYXJncz0iKyIsIGRlZmF1bHQ9WzBdKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWxhbWJkYXMiLCB0eXBlPWZsb2F0LCBuYXJncz0iKyIsIGRlZmF1bHQ9WzAuMCwgMC41LCAxLjAsIDIuMCwgNS4wXSkKICAgICMgZnVsbC1zY2FsZSBvdmVycmlkZXMgKENvbGFiKTogLS1lbmNvZGVyIHJvYmVydGEtYmFzZSAtLXN1YnNhbXBsZSAwIC0tZGV2aWNlIGN1ZGEgLi4uCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZW5jb2RlciIsIGRlZmF1bHQ9RU5DKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1heC1sZW5ndGgiLCB0eXBlPWludCwgZGVmYXVsdD1NQVhMRU4pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9RVBPQ0hTKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXN1YnNhbXBsZSIsIHR5cGU9aW50LCBkZWZhdWx0PVNVQiwgaGVscD0iMCA9IGZ1bGwgdHJhaW5pbmcgc2V0IikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kZXZpY2UiLCBkZWZhdWx0PU5vbmUsIGhlbHA9ImN1ZGEgfCBjcHUgfCBtcHMgKGRlZmF1bHQ6IGF1dG8pIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vb2QtcGFycXVldCIsIGRlZmF1bHQ9T09ELCBoZWxwPSJPT0QgcGFycXVldCBmb3IgdGhlIHZhcmlhbnQgY29tcGFyaXNvbiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcGVuYWx0eS1iYXRjaCIsIHR5cGU9aW50LCBkZWZhdWx0PVBFTkFMVFlfQkFUQ0gpCiAgICBhID0gYXAucGFyc2VfYXJncygpCiAgICBFTkMsIE1BWExFTiwgRVBPQ0hTLCBTVUIgPSBhLmVuY29kZXIsIGEubWF4X2xlbmd0aCwgYS5lcG9jaHMsIGEuc3Vic2FtcGxlCiAgICBPT0QsIERFVklDRV9PVkVSUklERSwgUEVOQUxUWV9CQVRDSCA9IGEub29kX3BhcnF1ZXQsIGEuZGV2aWNlLCBhLnBlbmFsdHlfYmF0Y2gKICAgIHByaW50KGYiZGV2aWNlPXtkZXYoKX0gZW5jb2Rlcj17RU5DfSBzdWJzYW1wbGU9e1NVQn0gZXBvY2hzPXtFUE9DSFN9IHBlbmFsdHlfYmF0Y2g9e1BFTkFMVFlfQkFUQ0h9IiwgZmx1c2g9VHJ1ZSkKICAgIGlmIGEubW9kZSBpbiAoInZhcmlhbnRzIiwgImJvdGgiKToKICAgICAgICBydW5fdmFyaWFudHMoYS5zZWVkcykKICAgIGlmIGEubW9kZSBpbiAoImxhbWJkYSIsICJib3RoIik6CiAgICAgICAgcnVuX2xhbWJkYShhLmxhbWJkYXMsIGEubGFtYmRhX3NlZWRzKQo=",
  'scripts/make_revision_figures.py': "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJGaWd1cmVzIGZvciB0aGUgcmV2aWV3ZXItcmV2aXNpb24gcmVzdWx0cywgaW4gdGhlIGhvdXNlIHN0eWxlIG9mIGZhaXRoZGV0ZWN0LnZpei4KCkdlbmVyYXRlcywgaW50byBwYXBlci9maWd1cmVzLzoKICBzbG90X2F0dHJpYnV0aW9uX21hc3MucG5nICAtLSBGVy9bRlVOQ10gc2xvdCBhdHRyaWJ1dGlvbiAoSUcgcGFkLWJhc2VsaW5lICsgb2NjbHVzaW9uKQogIGF0dGFja19zcGxpdC5wbmcgICAgICAgICAgIC0tIEZXIGF0dGFjayBkZWNvbXBvc2VkIChzd2FwIC8gZGVsZXRlIC8gZHVwbGljYXRlKQogIG9vZF90aHJlc2hvbGQucG5nICAgICAgICAgIC0tIE9PRCBGMSBnYXAgdnMgdGhyZXNob2xkLWZyZWUgUk9DLUFVQyBnYXAKQWxsIG51bWJlcnMgY29tZSBmcm9tIHJlc3VsdHMvKi5qc29uIChubyBoYXJkLWNvZGVkIGRhdGEpLgoiIiIKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgbWF0cGxvdGxpYgptYXRwbG90bGliLnVzZSgiQWdnIikKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdApmcm9tIG1hdHBsb3RsaWIucGF0Y2hlcyBpbXBvcnQgUGF0Y2gKCnBsdC5yY1BhcmFtcy51cGRhdGUoewogICAgImZpZ3VyZS5kcGkiOiAxMjAsICJzYXZlZmlnLmRwaSI6IDMwMCwgImZvbnQuc2l6ZSI6IDExLAogICAgImF4ZXMudGl0bGVzaXplIjogMTIsICJheGVzLnRpdGxld2VpZ2h0IjogImJvbGQiLCAiYXhlcy5sYWJlbHNpemUiOiAxMSwKICAgICJheGVzLnNwaW5lcy50b3AiOiBGYWxzZSwgImF4ZXMuc3BpbmVzLnJpZ2h0IjogRmFsc2UsCiAgICAiYXhlcy5ncmlkIjogVHJ1ZSwgImdyaWQuYWxwaGEiOiAwLjI1LCAibGVnZW5kLmZyYW1lb24iOiBGYWxzZSwKfSkKVk9SREVSID0gWyJiYXNlbGluZSIsICJzb2Z0cmVnIiwgImhhcmRtYXNrIl0KVkxBQkVMID0geyJiYXNlbGluZSI6ICJCYXNlbGluZSIsICJzb2Z0cmVnIjogIlNvZnRSZWciLCAiaGFyZG1hc2siOiAiSGFyZC1NYXNrIChGQUlUSCkifQpWQ09MT1IgPSB7ImJhc2VsaW5lIjogIiM2Yzc1N2QiLCAic29mdHJlZyI6ICIjZTA4MjE0IiwgImhhcmRtYXNrIjogIiMyYzdkNTkifQpPVVQgPSBQYXRoKCJwYXBlci9maWd1cmVzIikKCgpkZWYgX3NhdmUoZmlnLCBuYW1lKToKICAgIE9VVC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBwID0gT1VUIC8gbmFtZQogICAgZmlnLnNhdmVmaWcocCwgYmJveF9pbmNoZXM9InRpZ2h0IikKICAgIHBsdC5jbG9zZShmaWcpCiAgICBwcmludCgic2F2ZWQiLCBwKQoKCmRlZiBzbG90X21hc3MoKToKICAgIGQgPSBqc29uLmxvYWQob3BlbigicmVzdWx0cy9wbGFjZWhvbGRlcl9tYXNzLmpzb24iKSlbInZhcmlhbnRzIl0KICAgIGlnID0gW2Rbdl1bImlnX2Z3X3Bvc2l0aW9uX21hc3NfbWVhbiJdICogMTAwIGZvciB2IGluIFZPUkRFUl0KICAgIG9jYyA9IFtkW3ZdWyJvY2NsdXNpb25fZndfbWFzc19tZWFuIl0gKiAxMDAgZm9yIHYgaW4gVk9SREVSXQogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg2LjYsIDQuNCkpCiAgICB4ID0gbnAuYXJhbmdlKGxlbihWT1JERVIpKTsgdyA9IDAuMzgKICAgIGZvciBpLCB2IGluIGVudW1lcmF0ZShWT1JERVIpOgogICAgICAgIGF4LmJhcih4W2ldIC0gdyAvIDIsIGlnW2ldLCB3LCBjb2xvcj1WQ09MT1Jbdl0pCiAgICAgICAgYXguYmFyKHhbaV0gKyB3IC8gMiwgb2NjW2ldLCB3LCBjb2xvcj1WQ09MT1Jbdl0sIGFscGhhPTAuNSwgaGF0Y2g9Ii8vIiwgZWRnZWNvbG9yPSJ3aGl0ZSIpCiAgICAgICAgYXgudGV4dCh4W2ldIC0gdyAvIDIsIGlnW2ldICsgMSwgZiJ7aWdbaV06LjFmfSIsIGhhPSJjZW50ZXIiLCBmb250c2l6ZT04LjUsIGZvbnR3ZWlnaHQ9ImJvbGQiKQogICAgICAgIGF4LnRleHQoeFtpXSArIHcgLyAyLCBvY2NbaV0gKyAxLCBmIntvY2NbaV06LjFmfSIsIGhhPSJjZW50ZXIiLCBmb250c2l6ZT04LjUsIGZvbnR3ZWlnaHQ9ImJvbGQiKQogICAgYXguc2V0X3h0aWNrcyh4KTsgYXguc2V0X3h0aWNrbGFiZWxzKFtWTEFCRUxbdl0gZm9yIHYgaW4gVk9SREVSXSkKICAgIGF4LnNldF95bGFiZWwoIlNsb3QgYXR0cmlidXRpb24gbWFzcyAoJSkiKQogICAgYXguc2V0X3lsaW0oMCwgbWF4KG9jYyArIGlnKSArIDgpCiAgICBheC5zZXRfdGl0bGUoIkF0dHJpYnV0aW9uIG9uIGZ1bmN0aW9uLXdvcmQgU0xPVFMgKFtGVU5DXSBmb3IgSGFyZC1NYXNrKTpcbmlkZW50aXR5IGlzIGVyYXNlZCAoVGFibGUgMyAkPTAkKSBidXQgdGhlIHNsb3RzIGFyZSBzdGlsbCB1c2VkIikKICAgIGF4LmxlZ2VuZChoYW5kbGVzPVtQYXRjaChmYWNlY29sb3I9IiM4ODgiLCBsYWJlbD0iSW50ZWdyYXRlZCBHcmFkaWVudHMgKHBhZCBiYXNlbGluZSkiKSwKICAgICAgICAgICAgICAgICAgICAgICBQYXRjaChmYWNlY29sb3I9IiM4ODgiLCBhbHBoYT0wLjUsIGhhdGNoPSIvLyIsIGxhYmVsPSJPY2NsdXNpb24iKV0sCiAgICAgICAgICAgICAgZm9udHNpemU9OSwgbG9jPSJ1cHBlciByaWdodCIpCiAgICBfc2F2ZShmaWcsICJzbG90X2F0dHJpYnV0aW9uX21hc3MucG5nIikKCgpkZWYgYXR0YWNrX3NwbGl0KCk6CiAgICBhID0ganNvbi5sb2FkKG9wZW4oInJlc3VsdHMvYXR0YWNrX3NwbGl0Lmpzb24iKSlbInZhcmlhbnRzIl0KICAgIGNvbmRzID0gWyJjbGVhbiIsICJmd19zd2FwIiwgImZ3X2RlbGV0ZSIsICJmd19kdXBsaWNhdGUiXQogICAgY2xhYmVscyA9IFsiQ2xlYW4iLCAiU3dhcFxuKGlkZW50aXR5KSIsICJEZWxldGUiLCAiRHVwbGljYXRlIl0KICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oNy40LCA0LjQpKQogICAgeCA9IG5wLmFyYW5nZShsZW4oY29uZHMpKTsgdyA9IDAuOCAvIGxlbihWT1JERVIpCiAgICBheC5heHZzcGFuKDAuNSwgMS41LCBjb2xvcj0iI2YwZTZkMiIsIGFscGhhPTAuNiwgem9yZGVyPTApCiAgICBmb3IgaSwgdiBpbiBlbnVtZXJhdGUoVk9SREVSKToKICAgICAgICB2YWxzID0gW2Fbdl1bY11bImYxX21hY3JvIl0gKiAxMDAgZm9yIGMgaW4gY29uZHNdCiAgICAgICAgYXguYmFyKHggKyAoaSAtIChsZW4oVk9SREVSKSAtIDEpIC8gMikgKiB3LCB2YWxzLCB3LCBjb2xvcj1WQ09MT1Jbdl0sIGxhYmVsPVZMQUJFTFt2XSkKICAgIGF4LnNldF94dGlja3MoeCk7IGF4LnNldF94dGlja2xhYmVscyhjbGFiZWxzKQogICAgYXguc2V0X3lsaW0oNjAsIDEwMCk7IGF4LnNldF95bGFiZWwoIkYxIChtYWNybykiKQogICAgYXguc2V0X3RpdGxlKCJGdW5jdGlvbi13b3JkIGF0dGFjayBkZWNvbXBvc2VkIChwaWxvdDogZGlzdGlscm9iZXJ0YSwgc2VlZCAwKVxuSGFyZC1NYXNrIGlzIGZsYXQgdW5kZXIgc3dhcCAoY292ZXJlZCBieSB0aGUgcHJvb2YpLCBub3QgZGVsZXRlL2R1cGxpY2F0ZSIpCiAgICBheC50ZXh0KDEuMCwgNjEsICJwcm9vZiBjb3ZlcnNcbnRoaXMgY29sdW1uIiwgaGE9ImNlbnRlciIsIHZhPSJib3R0b20iLCBmb250c2l6ZT04LCBzdHlsZT0iaXRhbGljIiwgY29sb3I9IiM3YTVjMWUiKQogICAgYXgubGVnZW5kKGZvbnRzaXplPTksIG5jb2w9MywgbG9jPSJ1cHBlciBjZW50ZXIiLCBiYm94X3RvX2FuY2hvcj0oMC41LCAtMC4xMikpCiAgICBfc2F2ZShmaWcsICJhdHRhY2tfc3BsaXQucG5nIikKCgpkZWYgb29kX3RocmVzaG9sZCgpOgogICAgZiA9IGpzb24ubG9hZChvcGVuKCJyZXN1bHRzX2NvbGFiL3Jlc3VsdHMvZnVsbF9yZXN1bHRzLmpzb24iKSlbInZhcmlhbnRzIl0KICAgIGRlZiBhZ2codiwgbSk6CiAgICAgICAgYSA9IGZbdl1bIm9vZCJdWyJhZ2dyZWdhdGVkIl1bbV0KICAgICAgICByZXR1cm4gYVsibWVhbiJdICogMTAwLCAoYVsibWVhbiJdIC0gYVsibG8iXSkgKiAxMDAsIChhWyJoaSJdIC0gYVsibWVhbiJdKSAqIDEwMAogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg2LjgsIDQuNCkpCiAgICBncm91cHMgPSBbIk9PRCBGMSIsICJPT0QgUk9DLUFVQ1xuKCRcXHRpbWVzMTAwJCkiXQogICAgeCA9IG5wLmFyYW5nZShsZW4oZ3JvdXBzKSk7IHcgPSAwLjggLyBsZW4oVk9SREVSKQogICAgZm9yIGksIHYgaW4gZW51bWVyYXRlKFZPUkRFUik6CiAgICAgICAgZjEgPSBhZ2codiwgImYxX21hY3JvIik7IGF1ID0gYWdnKHYsICJyb2NfYXVjIikKICAgICAgICBtZWFucyA9IFtmMVswXSwgYXVbMF1dCiAgICAgICAgeWVyciA9IFtbZjFbMV0sIGF1WzFdXSwgW2YxWzJdLCBhdVsyXV1dCiAgICAgICAgYXguYmFyKHggKyAoaSAtIChsZW4oVk9SREVSKSAtIDEpIC8gMikgKiB3LCBtZWFucywgdywgeWVycj15ZXJyLCBjYXBzaXplPTMsCiAgICAgICAgICAgICAgIGNvbG9yPVZDT0xPUlt2XSwgbGFiZWw9VkxBQkVMW3ZdKQogICAgYXguc2V0X3h0aWNrcyh4KTsgYXguc2V0X3h0aWNrbGFiZWxzKGdyb3VwcykKICAgIGF4LnNldF95bGltKDAsIDEwMCk7IGF4LnNldF95bGFiZWwoIlNjb3JlIikKICAgIGF4LnNldF90aXRsZSgiT09EOiB0aGUgRjEgZ2FwIGlzIGxhcmdlbHkgYSB0aHJlc2hvbGQgYXJ0ZWZhY3Rcbih0aHJlc2hvbGQtZnJlZSBST0MtQVVDIGJhcmVseSBzZXBhcmF0ZXMgdGhlIHZhcmlhbnRzKSIpCiAgICBheC5hbm5vdGF0ZSgibGFyZ2UgRjFcbmdhcCIsIHh5PSgwLjAsIDQ3KSwgeHl0ZXh0PSgtMC4zNSwgMjUpLCBmb250c2l6ZT04LjUsIGNvbG9yPSIjYjAwIiwKICAgICAgICAgICAgICAgIGhhPSJjZW50ZXIiLCBhcnJvd3Byb3BzPWRpY3QoYXJyb3dzdHlsZT0iLT4iLCBjb2xvcj0iI2IwMCIpKQogICAgYXguYW5ub3RhdGUoIkFVQyBnYXBcbiRcXGFwcHJveCQgbm9pc2UiLCB4eT0oMS4wLCA3NiksIHh5dGV4dD0oMS4zNSwgNTUpLCBmb250c2l6ZT04LjUsIGNvbG9yPSIjMTc2IiwKICAgICAgICAgICAgICAgIGhhPSJjZW50ZXIiLCBhcnJvd3Byb3BzPWRpY3QoYXJyb3dzdHlsZT0iLT4iLCBjb2xvcj0iIzE3NiIpKQogICAgYXgubGVnZW5kKGZvbnRzaXplPTksIGxvYz0idXBwZXIgcmlnaHQiKQogICAgX3NhdmUoZmlnLCAib29kX3RocmVzaG9sZC5wbmciKQoKCmRlZiB2YXJpYW50X2NvbXBhcmlzb24oKToKICAgIGQgPSBqc29uLmxvYWQob3BlbigicmVzdWx0cy92YXJpYW50X2NvbXBhcmlzb24uanNvbiIpKVsidmFyaWFudHMiXQogICAgb3JkZXIgPSBbImJhc2VsaW5lIiwgImhhcmRtYXNrIiwgInJhbmRvbSIsICJkZWxldGlvbiJdCiAgICBsYWJlbHMgPSB7ImJhc2VsaW5lIjogIkJhc2VsaW5lIiwgImhhcmRtYXNrIjogIkhhcmQtTWFza1xuKFtGVU5DXSkiLAogICAgICAgICAgICAgICJyYW5kb20iOiAiUmFuZG9tXG5wbGFjZWhvbGRlciIsICJkZWxldGlvbiI6ICJEZWxldGlvbiJ9CiAgICBjb2xvcnMgPSB7ImJhc2VsaW5lIjogIiM2Yzc1N2QiLCAiaGFyZG1hc2siOiAiIzJjN2Q1OSIsICJyYW5kb20iOiAiIzk0NjdiZCIsICJkZWxldGlvbiI6ICIjYzA1MDRkIn0KICAgIGluZCA9IFtkW3ZdWyJpbmRvbWFpbl9mMSJdWyJtZWFuIl0gZm9yIHYgaW4gb3JkZXJdCiAgICBvb2QgPSBbZFt2XVsib29kX2YxIl1bIm1lYW4iXSBmb3IgdiBpbiBvcmRlcl0KICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oNy4wLCA0LjMpKQogICAgeCA9IG5wLmFyYW5nZShsZW4ob3JkZXIpKTsgdyA9IDAuMzgKICAgIGIxID0gYXguYmFyKHggLSB3IC8gMiwgaW5kLCB3LCBjb2xvcj1bY29sb3JzW3ZdIGZvciB2IGluIG9yZGVyXSwgbGFiZWw9IkluLWRvbWFpbiBGMSIpCiAgICBiMiA9IGF4LmJhcih4ICsgdyAvIDIsIG9vZCwgdywgY29sb3I9W2NvbG9yc1t2XSBmb3IgdiBpbiBvcmRlcl0sIGFscGhhPTAuNSwgaGF0Y2g9Ii8vIiwKICAgICAgICAgICAgICAgIGVkZ2Vjb2xvcj0id2hpdGUiLCBsYWJlbD0iT09EIEYxIikKICAgIGZvciB4aSwgdmkgaW4gemlwKHggLSB3IC8gMiwgaW5kKToKICAgICAgICBheC50ZXh0KHhpLCB2aSArIDEsIGYie3ZpOi4wZn0iLCBoYT0iY2VudGVyIiwgZm9udHNpemU9OC41LCBmb250d2VpZ2h0PSJib2xkIikKICAgIGZvciB4aSwgdmkgaW4gemlwKHggKyB3IC8gMiwgb29kKToKICAgICAgICBheC50ZXh0KHhpLCB2aSArIDEsIGYie3ZpOi4wZn0iLCBoYT0iY2VudGVyIiwgZm9udHNpemU9OC41LCBmb250d2VpZ2h0PSJib2xkIikKICAgIGF4LnNldF94dGlja3MoeCk7IGF4LnNldF94dGlja2xhYmVscyhbbGFiZWxzW3ZdIGZvciB2IGluIG9yZGVyXSkKICAgIGF4LnNldF95bGltKDAsIDEwMCk7IGF4LnNldF95bGFiZWwoIkYxIChtYWNybykiKQogICAgYXguc2V0X3RpdGxlKCJSZW1vdmluZyBmdW5jdGlvbiB3b3JkcyBieSBhbnkgcm91dGUga2VlcHMgaW4tZG9tYWluIEYxXG5idXQgY29zdHMgY3Jvc3MtZG9tYWluIHRyYW5zZmVyIChwaWxvdDogZGlzdGlscm9iZXJ0YSwgc2VlZCAwKSIpCiAgICBheC5sZWdlbmQoaGFuZGxlcz1bUGF0Y2goZmFjZWNvbG9yPSIjODg4IiwgbGFiZWw9IkluLWRvbWFpbiBGMSIpLAogICAgICAgICAgICAgICAgICAgICAgIFBhdGNoKGZhY2Vjb2xvcj0iIzg4OCIsIGFscGhhPTAuNSwgaGF0Y2g9Ii8vIiwgbGFiZWw9Ik9PRCBGMSIpXSwKICAgICAgICAgICAgICBmb250c2l6ZT05LCBuY29sPTIsIGxvYz0idXBwZXIgY2VudGVyIiwgYmJveF90b19hbmNob3I9KDAuNSwgLTAuMTApKQogICAgX3NhdmUoZmlnLCAidmFyaWFudF9jb21wYXJpc29uLnBuZyIpCgoKZGVmIGxhbWJkYV9zd2VlcCgpOgogICAgaW1wb3J0IG9zCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoInJlc3VsdHMvbGFtYmRhX3N3ZWVwLmpzb24iKToKICAgICAgICBwcmludCgic2tpcCBsYW1iZGFfc3dlZXAgKG5vIGpzb24geWV0KSIpOyByZXR1cm4KICAgIGQgPSBqc29uLmxvYWQob3BlbigicmVzdWx0cy9sYW1iZGFfc3dlZXAuanNvbiIpKTsgcyA9IGRbInN3ZWVwIl07IG1ldGEgPSBkLmdldCgibWV0YSIsIHt9KQogICAgZW5jID0gbWV0YS5nZXQoImVuY29kZXIiLCAiPyIpOyBucyA9IGxlbihtZXRhLmdldCgic2VlZHMiLCBbXSkgb3IgW10pCiAgICBzY2FsZSA9ICJmdWxsIGRhdGEiIGlmIG1ldGEuZ2V0KCJ0cmFpbl9zdWJzYW1wbGUiKSBpbiAoMCwgTm9uZSkgZWxzZSBmIm49e21ldGFbJ3RyYWluX3N1YnNhbXBsZSddfSIKICAgIGxhbSA9IFtyWyJsYW1iZGEiXSBmb3IgciBpbiBzXQogICAgZGVmIG1lKGtleSk6IHJldHVybiAoW3Jba2V5XVsibWVhbiJdIGZvciByIGluIHNdLAogICAgICAgICAgICAgICAgICAgICAgICAgW3Jba2V5XVsibWVhbiJdIC0gcltrZXldWyJsbyJdIGZvciByIGluIHNdLAogICAgICAgICAgICAgICAgICAgICAgICAgW3Jba2V5XVsiaGkiXSAtIHJba2V5XVsibWVhbiJdIGZvciByIGluIHNdKQogICAgZjEsIGYxbG8sIGYxaGkgPSBtZSgiaW5kb21haW5fZjEiKTsgc2UsIHNlbG8sIHNlaGkgPSBtZSgiaWRlbnRpdHlfc2Vuc2l0aXZpdHkiKQogICAgZmlnLCBheDEgPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oNi44LCA0LjMpKQogICAgYXgyID0gYXgxLnR3aW54KCkKICAgIGwxID0gYXgxLmVycm9yYmFyKGxhbSwgZjEsIHllcnI9W2YxbG8sIGYxaGldLCBmbXQ9Im8tIiwgY29sb3I9IiMyYzdkNTkiLCBjYXBzaXplPTMsIGxhYmVsPSJJbi1kb21haW4gRjEiKQogICAgbDIgPSBheDIuZXJyb3JiYXIobGFtLCBzZSwgeWVycj1bc2Vsbywgc2VoaV0sIGZtdD0icy0tIiwgY29sb3I9IiNlMDgyMTQiLCBjYXBzaXplPTMsCiAgICAgICAgICAgICAgICAgICAgICBsYWJlbD0iSWRlbnRpdHktc2Vuc2l0aXZpdHkgJHxcXERlbHRhIHB8JCAoJSkiKQogICAgYXgxLnNldF94bGFiZWwoIlNvZnRSZWcgcGVuYWx0eSB3ZWlnaHQgJFxcbGFtYmRhJCIpCiAgICBheDEuc2V0X3lsYWJlbCgiSW4tZG9tYWluIEYxIChtYWNybykiLCBjb2xvcj0iIzJjN2Q1OSIpCiAgICBheDIuc2V0X3lsYWJlbCgiSWRlbnRpdHktc2Vuc2l0aXZpdHkgKCUpIiwgY29sb3I9IiNlMDgyMTQiKQogICAgYXgxLnNldF90aXRsZShmIlNvZnRSZWcgJFxcbGFtYmRhJCBzd2VlcCAoe2VuY30sIHtzY2FsZX0sIHtuc30gc2VlZHMpOiBpbi1kb21haW4gRjEgaXMgc3RhYmxlXG4iCiAgICAgICAgICAgICAgICAgIGYidXAgdG8gJFxcbGFtYmRhe3s9fX0yJCB0aGVuIGNvbGxhcHNlczsgcmVsaWFuY2Ugc2hvd3Mgbm8gY2xlYW4gbW9ub3RvbmljIHRyZW5kIikKICAgIGF4MS5ncmlkKGFscGhhPTAuMjUpCiAgICBheDEubGVnZW5kKGhhbmRsZXM9W2wxLCBsMl0sIGZvbnRzaXplPTksIGxvYz0ibG93ZXIgbGVmdCIpCiAgICBfc2F2ZShmaWcsICJsYW1iZGFfc3dlZXAucG5nIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaW1wb3J0IHN5cwogICAgd2hpY2ggPSBzeXMuYXJndlsxOl0gb3IgWyJzbG90IiwgImF0dGFjayIsICJvb2QiLCAidmFyaWFudHMiLCAibGFtYmRhIl0KICAgIGlmICJzbG90IiBpbiB3aGljaDogc2xvdF9tYXNzKCkKICAgIGlmICJhdHRhY2siIGluIHdoaWNoOiBhdHRhY2tfc3BsaXQoKQogICAgaWYgIm9vZCIgaW4gd2hpY2g6IG9vZF90aHJlc2hvbGQoKQogICAgaWYgInZhcmlhbnRzIiBpbiB3aGljaDogdmFyaWFudF9jb21wYXJpc29uKCkKICAgIGlmICJsYW1iZGEiIGluIHdoaWNoOiBsYW1iZGFfc3dlZXAoKQo=",
}
for _p,_b in _SYNC.items():
    pathlib.Path(_p).parent.mkdir(parents=True, exist_ok=True)
    pathlib.Path(_p).write_bytes(base64.b64decode(_b))
os.system("find . -name __pycache__ -type d -exec rm -rf {} + 2>/dev/null")
print('synced', len(_SYNC), 'revised files')

In [ ]:
# 3) MAiDE-up CSV -> ./data/all_data.csv
import os, shutil
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/all_data.csv'):
    from google.colab import files
    print('Upload all_data.csv:'); up = files.upload()
    shutil.move(next(iter(up)), 'data/all_data.csv')
print('CSV ready:', os.path.exists('data/all_data.csv'))

## Experiment A — SoftReg $\lambda$ sweep (full scale)
Trains SoftReg at several $\lambda$ on roberta-base / full MAiDE-up / 3 seeds, and reports in-domain
F1 and function-word identity-sensitivity at each $\lambda$. No OOD needed, so this runs first.


In [ ]:
# A1) Run the lambda sweep (full scale, GPU). ~5-15 min on a T4.
!python scripts/run_desirable.py --mode lambda \
    --encoder roberta-base --max-length 256 --epochs 4 --subsample 0 --device cuda \
    --lambdas 0 0.25 0.5 1 2 4 --lambda-seeds 0 1 2 --penalty-batch 8

In [ ]:
# A2) Show the lambda-sweep result INLINE (this is what I read back from the notebook).
print(open('results/lambda_sweep.json').read())

In [ ]:
# A3) Lambda-sweep figure (saved to paper/figures/lambda_sweep.png) + display inline.
!python scripts/make_revision_figures.py lambda
from IPython.display import Image, display
display(Image('paper/figures/lambda_sweep.png'))

## Experiment B — full-scale same-domain, multi-generator
Adds same-domain reviews from held-in generators to training and evaluates on a disjoint set of
**held-out generators** (same domain, different generator) — this isolates generator shift from the
domain shift that confounds the main RAID OOD. Needs the RAID `reviews` data (downloaded next).


In [ ]:
# B1) Download RAID + cache the 'reviews' domain. Force a fresh cap so we get as much data as
#     this slice has, and print per-generator counts (the mixed-gen split needs enough rows).
import os, pandas as pd
from huggingface_hub import hf_hub_download
from faithdetect.data import load_raid_from_csv
RAID_CSV = hf_hub_download('liamdugan/raid', 'train.csv', repo_type='dataset')
if os.path.exists('results/cache/raid_reviews.parquet'):
    os.remove('results/cache/raid_reviews.parquet')   # force re-cap with the new cap below
ood = load_raid_from_csv(RAID_CSV, domains=('reviews',), cap_per_group=1000,
                         cache_path='results/cache/raid_reviews.parquet')
print('reviews rows:', len(ood),
      '| humans:', int((ood['label']==0).sum()), '| AI:', int((ood['label']==1).sum()))
print('AI rows per generator:'); print(ood[ood['label']==1]['model'].astype(str).value_counts())

In [ ]:
# B2) Auto-pick the held-in / held-out AI generators from whatever is actually present
#     (>= MIN rows each), so the split can't silently mismatch the data.
import pandas as pd
g = pd.read_parquet('results/cache/raid_reviews.parquet')
MIN = 25                                   # min AI rows for a generator to be usable
vc = g[g['label']==1]['model'].astype(str).value_counts()
gens = [m for m, c in vc.items() if c >= MIN and m.lower() != 'human']
held_in, held_out = gens[0::2], gens[1::2]  # alternate -> balanced counts on each side
print('usable AI generators (>= %d rows):' % MIN); print(vc[vc >= MIN])
print('HELD_IN :', held_in)
print('HELD_OUT:', held_out)
assert held_in and held_out, ('Not enough generators for a split in this RAID slice — '
                              'raise cap_per_group in B1, lower MIN, or accept the pilot result.')
HELD_IN, HELD_OUT = ' '.join(held_in), ' '.join(held_out)

In [ ]:
# B3) Mixed-generator training at full scale, using the auto-detected split from B2.
#     (If B2 raised an AssertionError, the RAID reviews slice is too small for this experiment.)
!python scripts/run_grid.py --name mixedgen --encoder roberta-base --train_device cuda \
    --xai_device cuda --seeds 0 1 2 --variants baseline hardmask softreg \
    --epochs 4 --batch_size 16 --max_length 256 --train_subsample 0 \
    --train_mix_parquet results/cache/raid_reviews.parquet \
    --train_mix_generators {HELD_IN} \
    --heldout_generators {HELD_OUT} \
    --no_leakage --out results/mixedgen_results.json --figdir figures

In [ ]:
# B4) Show the mixed-generator result INLINE (held-out-generator F1 per variant).
import json
m = json.load(open('results/mixedgen_results.json'))
mc = m.get('meta',{}).get('config', {})
print('train_mix_generators:', mc.get('train_mix_generators'))
print('heldout_generators :', mc.get('heldout_generators'))
for v in ('baseline','hardmask','softreg'):
    V = m['variants'].get(v, {})
    ind = V.get('indomain',{}).get('aggregated',{}).get('f1_macro',{})
    ho  = V.get('heldout_gen',{}).get('aggregated',{}).get('f1_macro',{})
    print(f"{v:9s} in-domain F1={ind.get('mean')}  held-out-gen F1={ho.get('mean')}")
print('--- full JSON below ---')
print(json.dumps(m, indent=1)[:6000])

In [ ]:
# B5) Held-out-generator figure (made by run_grid) + display inline.
from IPython.display import Image, display
import os
for p in ['figures/06c_heldout_generator.png','figures/06b_cross_generator.png']:
    if os.path.exists(p): display(Image(p))

## Bundle for download (backup) — and you're done
Sending the executed notebook back is enough (results are printed inline above). This cell also
saves a small zip you can download as a backup.


In [ ]:
# Bundle the two JSONs + the new figures, and offer a download.
import zipfile, os
keep = ['results/lambda_sweep.json','results/mixedgen_results.json',
        'paper/figures/lambda_sweep.png','figures/06c_heldout_generator.png']
with zipfile.ZipFile('faithdetect_revision_results.zip','w') as z:
    for p in keep:
        if os.path.exists(p): z.write(p)
print('zipped:', [p for p in keep if os.path.exists(p)])
from google.colab import files; files.download('faithdetect_revision_results.zip')